# Proyecto: Pronóstico de demanda de picking – CEDI
## Notebook 03 · Modelado y evaluación

**Equipo:** Natasha Orozco · Xiomy Baquero · Andrés Zabala

### Continuidad del proyecto
- **Notebook 01 (EDA):** describió la demanda de picking (líneas por día) en los canales Picking Packing y TAT. Encontró patrón semanal, efectos fuertes de festivos y eventos, y cuatro cambios de régimen por decisiones de negocio.
- **Notebook 02 (Feature Engineering):** construyó 23 variables disponibles al corte del viernes y comprobó que no hay data leakage (0 fallas en 201 cortes por canal).

### Objetivo de este notebook
Elegir el modelo que mejor pronostica las **líneas de picking por día y canal** de la semana siguiente, con corte el viernes en la noche y un horizonte de 1 a 5 días. Se entrena **un modelo por canal**.

### Modelos que se comparan
| Modelo | Papel |
|---|---|
| Naive estacional (`lag_7`) | Línea base: "lo mismo que el mismo día de la semana pasada" |
| Promedio de 4 semanas | Línea base: el mejor predictor simple del EDA |
| Prophet con regresores | Modelo central de series de tiempo |
| Regresión con errores ARIMA | Modelo central de series de tiempo (SARIMAX sin parte estacional) |
| Gradient Boosting con rezagos | Contraste de machine learning |

**Regla de decisión:** gana el modelo con **menor WAPE en may-nov 2026**, siempre que no subestime de forma importante. Si ninguno le gana con claridad al promedio de 4 semanas, esa es la recomendación.

### Estructura
1. Configuración y carga de datos
2. Métricas de evaluación
3. Diseño del backtesting
4. Experimento 0: con y sin variables del mismo día
5. Líneas base
6. Prophet con regresores
7. Regresión con errores ARIMA
8. Gradient Boosting con rezagos
9. Prueba de las variables candidatas
10. Comparación final y elección del modelo
11. Pronóstico de la semana del 23 al 27 de noviembre en operarios
12. Conclusiones, limitaciones y próximos pasos

## 1. Configuración y carga de datos

Se cargan los dos archivos que exportó el notebook 02:
- `features_modelo.csv`: días con objetivo observado, para entrenar y validar.
- `features_proxima_semana.csv`: variables de la semana del 23 al 27 de noviembre de 2026, sin objetivo, para el pronóstico real.

Las listas de variables (`FEATURES`) son **las mismas que se auditaron** en el notebook 02. Las validaciones con `assert` detienen el notebook si los archivos no coinciden con los que se exportaron.

In [1]:
# ============================================
# CELDA 1: Librerías, constantes y carga de datos
# ============================================
# !pip install -q prophet statsmodels        # en Colab: Prophet y statsmodels ya vienen; descomenta si faltan
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import warnings
warnings.filterwarnings('ignore')

pio.templates.default = 'plotly_white'
COL = {'Picking Packing': '#2a78d6', 'TAT': '#eb6834'}          # mismos colores del EDA y del notebook 02

CARPETA = '/content/'                                            # en Colab, sube aquí los archivos

# --- Constantes heredadas del notebook 02 ---
INICIO_42H    = pd.Timestamp('2026-07-15')     # régimen actual: lunes a viernes
INICIO_EVAL   = pd.Timestamp('2026-05-01')     # ventana de evaluación principal: may-nov 2026
OBJETIVO      = 'Lineas'
METADATOS     = ['Fecha', 'Canal', 'corte', 'horizonte_dias']

# Variables finales del notebook 02 (sección 9) – auditadas sin data leakage
COMUNES = ['dia_semana', 'mes', 'dia_despues_festivo', 'dias_operativos_semana', 'es_quincena',
           'semana_santa', 'semana_post_semana_santa', 'pre_cierre_anio', 'cierre_anio', 'reactivacion_anio',
           'catalogo_2026', 'jornada_42h',
           'lag_7', 'lag_14', 'lag_21', 'lag_28', 'prom_4_semanas', 'lag_364', 'nivel_reciente', 'n_lags_disponibles']
FEATURES = {'Picking Packing': COMUNES,
            'TAT': COMUNES + ['dia_mes', 'expansion_tat_2024', 'recorte_rutas_tat']}
CANDIDATAS = {'Picking Packing': ['es_quincena', 'dias_operativos_semana'],
              'TAT': ['es_quincena', 'dias_operativos_semana', 'dia_mes']}

# Productividad (EDA): líneas que alista un operario por día → convierte el pronóstico en personal
LINEAS_POR_OPERARIO = {'Picking Packing': 106, 'TAT': 249}

# --- Carga ---
datos = pd.read_csv(CARPETA + 'features_modelo.csv', parse_dates=['Fecha', 'corte'])
proxima = pd.read_csv(CARPETA + 'features_proxima_semana.csv', parse_dates=['Fecha', 'corte'])

# --- Validaciones contra el notebook 02: si algo no coincide, el notebook se detiene aquí ---
TODAS = list(dict.fromkeys(FEATURES['Picking Packing'] + FEATURES['TAT']))
assert datos.shape == (2_203, 28),                         'features_modelo.csv no coincide con el notebook 02'
assert set(datos.columns) == set(METADATOS + TODAS + [OBJETIVO]), 'Las columnas no coinciden'
assert datos[OBJETIVO].notna().all() and (datos[OBJETIVO] > 0).all(), 'Hay objetivos vacíos o en cero'
assert not datos.duplicated(['Fecha', 'Canal']).any(),     'Hay fechas repetidas por canal'
assert (datos['corte'] < datos['Fecha']).all(),            'Hay filas con corte posterior a la fecha'
assert (datos['corte'].dt.dayofweek == 6).all(),           'El corte no es domingo (cierre de semana)'
assert proxima.shape == (10, 27) and proxima['Fecha'].min() > datos['Fecha'].max(), 'Revisar la próxima semana'

datos = datos.sort_values(['Canal', 'Fecha']).reset_index(drop=True)

# --- Resumen ---
print(f'features_modelo.csv: {datos.shape[0]:,} filas × {datos.shape[1]} columnas ✓\n')
resumen = datos.groupby('Canal').agg(
    Filas=('Fecha', 'size'), Desde=('Fecha', 'min'), Hasta=('Fecha', 'max'),
    Semanas=('corte', 'nunique'),
    Filas_eval_may_nov=('Fecha', lambda s: (s >= INICIO_EVAL).sum()),
    Filas_regimen_42h=('Fecha', lambda s: (s >= INICIO_42H).sum()),
    Lineas_mediana=(OBJETIVO, 'median'))
resumen['Variables'] = [len(FEATURES[c]) for c in resumen.index]
print(resumen.to_string())

print(f'\nfeatures_proxima_semana.csv: {proxima.Fecha.min().date()} a {proxima.Fecha.max().date()} '
      f'(corte {proxima.corte.iloc[0].date()}), {proxima.shape[0]} filas = 5 días × 2 canales ✓')

features_modelo.csv: 2,203 filas × 28 columnas ✓

                 Filas      Desde      Hasta  Semanas  Filas_eval_may_nov  Filas_regimen_42h  Lineas_mediana  Variables
Canal                                                                                                                  
Picking Packing   1108 2023-01-02 2026-11-20      203                 145                 87           702.0         20
TAT               1095 2023-01-02 2026-11-20      201                 144                 87          1874.0         23

features_proxima_semana.csv: 2026-11-23 a 2026-11-27 (corte 2026-11-22), 10 filas = 5 días × 2 canales ✓


### Conclusiones: carga de datos
- Los archivos coinciden con los que exportó el notebook 02: **2,203 filas** (1,108 de PP y 1,095 de TAT) del 2-ene-2023 al 20-nov-2026, sin objetivos vacíos ni fechas repetidas.
- Hay **≈200 semanas** con corte por canal: son los posibles puntos de corte del backtesting.
- **Ventana de evaluación principal (may-nov 2026):** 145 días en PP y 144 en TAT. De ellos, **87 por canal** son del régimen actual (jornada de 42 h, lunes a viernes). Es poca historia, así que la evaluación debe cubrir toda la ventana de mayo a noviembre y no solo el régimen actual.
- **Próxima semana:** 5 días operativos (23 al 27 de noviembre) por canal, con todas las variables ya calculadas al corte del domingo 22.

## 2. Métricas de evaluación

En la sesión 5 se vieron las métricas de regresión **MAE, MSE/RMSE, R² y MAPE**. Se reportan todas, junto con dos métricas propias del problema: **WAPE** y **sesgo**.

| Métrica | Lectura | Uso en el proyecto |
|---|---|---|
| **WAPE** | % del volumen total que se pronosticó mal | **Principal**: se traduce directo a operarios |
| MAE | Líneas de error por día | Complementaria, en unidades del negocio |
| RMSE | Castiga más los errores grandes | Detecta modelos que fallan mucho en días puntuales |
| R² | % de la variación explicada | Referencia; es engañosa en series con niveles cambiantes |
| MAPE | Error % promedio por día | **No se usa para decidir** (ver la demostración) |
| **Sesgo** | + sobreestima · − subestima | **Criterio de negocio**: subestimar deja la operación corta de personal |

**¿Por qué WAPE y no MAPE?** El MAPE divide el error de **cada día** por el volumen real de ese día. En un día con muy pocas líneas (Semana Santa, cierre de año), un error pequeño en líneas se vuelve un error porcentual enorme y domina el promedio. El WAPE divide el error total entre el volumen total: un día de 5 líneas pesa lo que pesan 5 líneas.

**Sobreajuste (sesión 5):** en los modelos que aprenden de los datos (sobre todo el Gradient Boosting), se compara el error de **entrenamiento** con el de **validación**. Una brecha grande indica que el modelo memorizó el ruido.

In [2]:
# ============================================
# CELDA 2: Métricas de evaluación
# ============================================
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, mean_absolute_percentage_error)

def metricas(real, pred):
    """Métricas de regresión de la sesión 5 (MAE, RMSE, R², MAPE) + WAPE y sesgo del proyecto."""
    real, pred = np.asarray(real, float), np.asarray(pred, float)
    ok = ~np.isnan(pred)                                   # un modelo sin pronóstico para un día no se evalúa ahí
    real, pred = real[ok], pred[ok]
    return pd.Series({
        'WAPE_%':  np.abs(real - pred).sum() / real.sum() * 100,          # principal
        'MAE':     mean_absolute_error(real, pred),
        'RMSE':    np.sqrt(mean_squared_error(real, pred)),
        'R2':      r2_score(real, pred),
        'MAPE_%':  mean_absolute_percentage_error(real, pred) * 100,
        'Sesgo_%': (pred - real).sum() / real.sum() * 100,                # + sobreestima · − subestima
        'n':       len(real),
    })

# --- Demostración: ¿por qué WAPE y no MAPE? ---
# Pronóstico de prueba: el promedio de las 4 semanas anteriores (solo para ilustrar las métricas;
# la evaluación formal de las líneas base está en la sección 5).
demo = datos.dropna(subset=['prom_4_semanas'])
filas = []
for canal in COL:
    d = demo[demo.Canal == canal]
    for nombre, mascara in [('Todos los días', d.Fecha.notna()),
                            ('Sin cierre de año', d.cierre_anio == 0),
                            ('Solo cierre de año', d.cierre_anio == 1)]:
        m = metricas(d.loc[mascara, OBJETIVO], d.loc[mascara, 'prom_4_semanas'])
        filas.append({'Canal': canal, 'Días': nombre, **m.round(1).to_dict()})
tabla_demo = pd.DataFrame(filas).set_index(['Canal', 'Días'])
tabla_demo['n'] = tabla_demo['n'].astype(int)
print('Métricas de un pronóstico simple (promedio de 4 semanas), 2023-2026\n')
print(tabla_demo.to_string())

# --- Un día concreto: el error porcentual explota cuando el volumen real es bajo ---
peor = demo.assign(error_pct=lambda x: (x.prom_4_semanas - x[OBJETIVO]).abs() / x[OBJETIVO] * 100) \
           .nlargest(3, 'error_pct')[['Fecha', 'Canal', OBJETIVO, 'prom_4_semanas', 'error_pct']]
print('\nLos 3 días con mayor error porcentual:')
print(peor.round(0).to_string(index=False))

Métricas de un pronóstico simple (promedio de 4 semanas), 2023-2026

                                    WAPE_%     MAE    RMSE   R2  MAPE_%  Sesgo_%     n
Canal           Días                                                                  
Picking Packing Todos los días        27.6   193.9   259.9  0.3    73.7     -1.0  1102
                Sin cierre de año     25.7   183.5   241.0  0.4    55.8     -3.2  1075
                Solo cierre de año   221.6   608.6   667.0 -8.1   786.3    221.6    27
TAT             Todos los días        19.7   376.0   559.4  0.0   388.3     -0.4  1089
                Sin cierre de año     19.0   363.1   531.7  0.1   267.6     -1.1  1073
                Solo cierre de año    92.2  1239.9  1529.2 -0.8  8480.3     66.4    16

Los 3 días con mayor error porcentual:
     Fecha Canal  Lineas  prom_4_semanas  error_pct
2024-03-27   TAT     1.0          1998.0   199750.0
2024-01-02   TAT     2.0          1740.0    86925.0
2025-04-16   TAT     5.0          1927.

### Conclusiones: métricas
- **El MAPE no sirve para decidir en este problema.** En TAT da **388%** con todos los días, aunque el pronóstico falla solo el **20% del volumen** (WAPE). Tres días casi sin operación (1, 2 y 5 líneas) generan errores de **38,000% a 200,000%** cada uno y dominan el promedio.
- **El WAPE es estable y se lee en lenguaje de negocio:** "el pronóstico se equivoca en el X% de las líneas de la semana". Por eso es la métrica principal.
- **El R² también engaña aquí:** en TAT da **0.0** aunque el WAPE es de 20%. Hay mucha variación de día a día alrededor de un nivel estable, y el R² la penaliza. Se reporta solo como referencia.
- **Los días de cierre de año se rompen con cualquier pronóstico ingenuo:** WAPE de 222% en PP, con un sesgo de **+222%** (sobreestima porque copia semanas normales). Ahí es donde deben aportar las variables de eventos del notebook 02.
- **Sesgo:** el promedio de 4 semanas casi no tiene sesgo en el agregado (−1% PP, −0.4% TAT). Cualquier modelo que se proponga debe mantenerlo cerca de cero, y en ningún caso con una subestimación importante.
- **Primera referencia:** sin cierre de año, el promedio de 4 semanas se equivoca en ≈**26% del volumen en PP** y ≈**19% en TAT**. Es la marca que los modelos deben superar (se confirma con el backtesting en la sección 5).

## 3. Diseño del backtesting

**Backtesting:** se simula la rutina de cada viernes en el pasado. En cada corte, el modelo se entrena **solo con la historia conocida hasta ese día** y pronostica los días operativos de la semana siguiente. Después se compara contra lo que realmente pasó y se avanza una semana.

**¿Por qué no `train_test_split`?** En una serie de tiempo el orden importa. Una partición aleatoria dejaría al modelo entrenar con días posteriores a los que pronostica (data leakage) y el error saldría artificialmente bajo.

| Elemento | Decisión | Por qué |
|---|---|---|
| Punto de corte | Domingo de cada semana (columna `corte`) | Es el cierre del viernes; viene auditado del notebook 02 |
| Qué se pronostica | Días operativos de la semana siguiente (horizonte 1-5 días; 1-6 antes de la jornada de 42 h) | Es la decisión real de la operación |
| Con qué se entrena | **Toda la historia hasta el corte** (ventana creciente) | Los festivos y eventos ocurren pocas veces: el modelo necesita los años anteriores para aprenderlos |
| Qué se califica | Semanas completas de **mayo a noviembre de 2026** | Es el periodo más parecido a la operación actual |
| Cómo se resume | Una tabla larga con un pronóstico por día, y las métricas de la sección 2 por grupo | Todos los modelos se comparan con los mismos días y la misma regla |

**Una sola función para todos los modelos.** `backtest()` recibe el modelo como parámetro. Cada modelo es una función `modelo(train, test, canal)` que devuelve el pronóstico de la semana. Así, ningún modelo se evalúa con reglas distintas.

In [13]:
# ============================================
# CELDA 3: Diseño del backtesting por cortes semanales
# ============================================
import time

def cortes_evaluacion(canal, desde=INICIO_EVAL, hasta=None, paso=1):        # ← NUEVO: hasta y paso
    """Cortes (cierre del viernes) desde el inicio de la ventana: solo semanas completas dentro de ella.
    hasta y paso permiten usar otros cortes (p. ej., para elegir parámetros ANTES de la evaluación)."""
    d = datos[datos.Canal == canal]
    c = np.sort(d.loc[d.corte >= desde, 'corte'].unique())
    if hasta is not None:                                                    # ← NUEVO
        c = c[c < np.datetime64(hasta)]                                      # ← NUEVO: solo cortes antes de `hasta`
    return c[::paso]                                                         # ← NUEVO: 1 de cada `paso` semanas

def backtest(nombre, modelo, canal, desde=INICIO_EVAL, hasta=None, paso=1):  # ← NUEVO: hasta y paso
    """Reproduce la rutina de cada viernes: entrena con lo conocido hasta el corte y pronostica la semana siguiente.

    modelo(train, test, canal) debe devolver:
      - un arreglo con el pronóstico de cada fila de `test`, o
      - una tupla (pronóstico de test, pronóstico de train) para medir el sobreajuste.
    """
    d = datos[datos.Canal == canal]
    filas, errores_train = [], []
    inicio = time.time()
    for corte in cortes_evaluacion(canal, desde, hasta, paso):               # ← NUEVO: pasa hasta y paso
        train = d[d.Fecha <= corte]                  # ventana creciente: TODA la historia hasta el corte
        test = d[d.corte == corte]                   # días operativos de la semana siguiente
        assert train.Fecha.max() < test.Fecha.min(), 'El entrenamiento incluye días de la semana pronosticada'

        salida = modelo(train, test, canal)
        pred, pred_train = salida if isinstance(salida, tuple) else (salida, None)

        filas.append(test[METADATOS + [OBJETIVO]].assign(modelo=nombre, pred=np.asarray(pred, float)))
        if pred_train is not None:                   # error dentro de la muestra (sesión 5: sobreajuste)
            errores_train.append(metricas(train[OBJETIVO], pred_train)['WAPE_%'])

    res = pd.concat(filas, ignore_index=True).rename(columns={OBJETIVO: 'real'})
    res.attrs['wape_train'] = np.mean(errores_train) if errores_train else np.nan
    res.attrs['segundos'] = time.time() - inicio
    return res

def resumen(resultados, por=('modelo', 'Canal')):
    """Métricas por grupo sobre la tabla larga de pronósticos."""
    return (resultados.groupby(list(por), sort=False)
                      .apply(lambda g: metricas(g['real'], g['pred'])).round(1))

# --- Los cortes de la ventana de evaluación ---
for canal in COL:
    c = cortes_evaluacion(canal)
    dias = datos[(datos.Canal == canal) & datos.corte.isin(c)].groupby('corte').size()
    print(f'{canal:16s}: {len(c)} cortes · del {pd.Timestamp(c[0]).date()} al {pd.Timestamp(c[-1]).date()} · '
          f'{dias.sum()} días a pronosticar · días por semana: {dias.value_counts().sort_index().to_dict()}')

# --- Prueba 1: un modelo "tramposo" que devuelve el valor real debe dar error 0 ---
tramposo = lambda train, test, canal: test[OBJETIVO].to_numpy()
assert resumen(backtest('tramposo', tramposo, 'TAT'))['WAPE_%'].iloc[0] == 0
print('\nPrueba 1 ✓ Un pronóstico perfecto da WAPE = 0 (la función mide bien)')

# --- Prueba 2: un modelo muy simple, para ver la función completa funcionando ---
def media_historica(train, test, canal):
    """Pronostica el promedio de TODA la historia del canal. Devuelve también el ajuste en train."""
    media = train[OBJETIVO].mean()
    return np.full(len(test), media), np.full(len(train), media)

partes = [backtest('Media histórica', media_historica, c) for c in COL]
resultados_demo = pd.concat(partes, ignore_index=True)
print('Prueba 2 ✓ Backtesting completo con un modelo de prueba')
for canal, p in zip(COL, partes):
    print(f'  {canal:16s}: {p.attrs["segundos"]:.1f} s · WAPE promedio en entrenamiento: {p.attrs["wape_train"]:.1f}%')
print('\nTabla larga de pronósticos (primeras filas):')
print(resultados_demo.head(6).round(0).to_string(index=False))
print('\nResumen por canal (may-nov 2026):')
print(resumen(resultados_demo).to_string())

Picking Packing : 29 cortes · del 2026-05-03 al 2026-11-15 · 144 días a pronosticar · días por semana: {4: 7, 5: 16, 6: 6}
TAT             : 29 cortes · del 2026-05-03 al 2026-11-15 · 143 días a pronosticar · días por semana: {4: 7, 5: 17, 6: 5}

Prueba 1 ✓ Un pronóstico perfecto da WAPE = 0 (la función mide bien)
Prueba 2 ✓ Backtesting completo con un modelo de prueba
  Picking Packing : 0.1 s · WAPE promedio en entrenamiento: 38.1%
  TAT             : 0.1 s · WAPE promedio en entrenamiento: 21.6%

Tabla larga de pronósticos (primeras filas):
     Fecha           Canal      corte  horizonte_dias   real          modelo  pred
2026-05-04 Picking Packing 2026-05-03               1   32.0 Media histórica 696.0
2026-05-05 Picking Packing 2026-05-03               2 1146.0 Media histórica 696.0
2026-05-06 Picking Packing 2026-05-03               3  947.0 Media histórica 696.0
2026-05-07 Picking Packing 2026-05-03               4 1044.0 Media histórica 696.0
2026-05-08 Picking Packing 2026-05-

### Conclusiones: diseño del backtesting
- **29 cortes por canal** (del 3-may al 15-nov-2026) y **≈144 días a pronosticar** en cada canal. Son las mismas semanas y los mismos días para todos los modelos.
- **Las semanas no tienen todas el mismo tamaño:** 7 semanas de 4 días (con festivo), 16-17 de 5 días y 5-6 de 6 días (con sábado, antes de la jornada de 42 h). Hasta julio, el horizonte llega a 6 días (el sábado).
- **La función está validada:** un pronóstico perfecto da WAPE = 0, y el `assert` impide que el entrenamiento incluya días de la semana pronosticada.
- **El modelo de prueba ya deja una lección:** el promedio de toda la historia **sobreestima TAT en +32%**. Su error de entrenamiento (22%) es mucho menor que el de validación (36.5%), pero **no por sobreajuste**: es porque el nivel de TAT bajó con el recorte de rutas de mayo, y un promedio de 4 años no se entera. Los modelos que se comparan deben reaccionar a los **cambios de régimen**, ya sea con los rezagos recientes o con las variables de régimen.
- **Siguiente paso:** las líneas base (naive y promedio de 4 semanas) son la primera marca real que deben superar los modelos.

## 4. Experimento 0: ¿por qué no se usan las variables del mismo día?

En el EDA (sección 24) se descartaron las variables operativas del mismo día: **puntos de envío, pedidos, SKUs, ubicaciones, ruteos y operarios**. Explican casi perfecto el volumen (correlaciones de hasta 0.99), pero **solo se conocen después de que ocurre el picking**. El viernes en la noche nadie sabe cuántos pedidos llegarán el miércoles siguiente.

Este experimento **cuantifica la trampa**: se entrena el **mismo modelo** dos veces, con el mismo backtesting de la sección 3.

| Versión | Variables |
|---|---|
| **Honesto** | Calendario, eventos, régimen y el conjunto histórico lineal (`prom_4_semanas`, `lag_364`, `nivel_reciente`) |
| **Con trampa** | Las mismas + las 6 variables operativas del mismo día |

**Modelo:** regresión **Ridge** (regresión lineal con regularización, sesión 5) dentro de un **pipeline**:
- `dia_semana` y `mes` → **OneHotEncoder**: son categorías sin orden lineal (el martes no es "el doble" del lunes).
- Variables numéricas → **imputación con la mediana** (`lag_364` no existe en 2023) + **StandardScaler**, porque Ridge penaliza los coeficientes y necesita todas las variables en la misma escala.

**Ojo:** este experimento **no compite** en la selección de modelos. Su único fin es demostrar por qué el modelo con trampa no se puede usar, por bueno que parezca.

In [5]:
# ============================================
# CELDA 4: Experimento 0 – ¿cuánto "mejora" el modelo si hace trampa?
# ============================================
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

# --- 1. Variables operativas del MISMO día (dataset original, misma lectura del EDA) ---
CANALES = {'Ruteo Picking Packing': 'Picking Packing', 'Ruteo TAT': 'TAT'}
MISMO_DIA = ['Puntos_Envio_Distintos', 'Pedidos_Distintos', 'SKUs_Distintos',
             'Ubicaciones_Visitadas', 'Ruteos_Distintos', 'Usuarios_Picking']

crudo = pd.read_csv(CARPETA + 'dataset_version3.csv', sep=';', decimal=',', encoding='utf-8-sig', low_memory=False)
crudo['Fecha'] = pd.to_datetime(crudo['Fecha'], dayfirst=True)
crudo['Canal'] = crudo['Canal'].map(CANALES)
assert len(crudo) == 35_221, 'El dataset no coincide con el del EDA'

operativas = crudo.groupby(['Fecha', 'Canal'], as_index=False)[MISMO_DIA].sum()   # total del día por canal
datos_exp0 = datos.merge(operativas, on=['Fecha', 'Canal'], how='left')
assert datos_exp0[MISMO_DIA].notna().all().all(), 'Hay días del modelo sin variables operativas'

# --- 2. El MISMO modelo para las dos versiones: regresión Ridge con preprocesamiento ---
CATEGORICAS = ['dia_semana', 'mes']                          # sin orden lineal → indicadoras (one-hot)
HONESTAS = [v for v in COMUNES if v not in CATEGORICAS + ['lag_7', 'lag_14', 'lag_21', 'lag_28',
                                                          'n_lags_disponibles']]   # conjunto lineal (VIF ≤ 1.5)
EXTRA_TAT = ['dia_mes', 'expansion_tat_2024', 'recorte_rutas_tat']

def ridge_pipeline(numericas):
    prep = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAS),
        ('num', Pipeline([('imputar', SimpleImputer(strategy='median')),   # lag_364 no existe en 2023
                          ('escalar', StandardScaler())]), numericas)])
    return Pipeline([('prep', prep), ('modelo', Ridge(alpha=1.0))])

def hacer_ridge(con_trampa):
    def modelo(train, test, canal):
        numericas = HONESTAS + (EXTRA_TAT if canal == 'TAT' else []) + (MISMO_DIA if con_trampa else [])
        cols = CATEGORICAS + numericas
        p = ridge_pipeline(numericas).fit(train[cols], train[OBJETIVO])
        return p.predict(test[cols]), p.predict(train[cols])
    return modelo

# --- 3. Backtesting con los mismos cortes de la sección 3 ---
datos_original = datos
datos = datos_exp0                                            # backtest() lee la variable global `datos`
partes = [backtest(nombre, hacer_ridge(trampa), canal)
          for canal in COL
          for nombre, trampa in [('Ridge honesto', False), ('Ridge con trampa', True)]]
datos = datos_original                                        # se restaura: las operativas no se usan más

exp0 = pd.concat(partes, ignore_index=True)
tabla_exp0 = resumen(exp0, por=('Canal', 'modelo'))
tabla_exp0['WAPE_train_%'] = [round(p.attrs['wape_train'], 1) for p in partes]
print('Experimento 0 · Ridge, backtesting may-nov 2026 (29 cortes por canal)\n')
print(tabla_exp0[['WAPE_%', 'WAPE_train_%', 'MAE', 'RMSE', 'R2', 'Sesgo_%', 'n']].to_string())

# --- 4. Gráfica: el modelo con trampa "sigue" la realidad casi perfecto ---
canal = 'Picking Packing'
g = exp0[(exp0.Canal == canal) & (exp0.Fecha >= '2026-09-01')]
fig = go.Figure()
fig.add_scatter(x=g.Fecha.unique(), y=g.drop_duplicates('Fecha').real, name='Real',
                line=dict(color='#333333', width=2.5))
for nombre, color, dash in [('Ridge honesto', COL[canal], 'solid'), ('Ridge con trampa', '#b0b0b0', 'dot')]:
    s = g[g.modelo == nombre]
    fig.add_scatter(x=s.Fecha, y=s.pred, name=nombre, line=dict(color=color, width=2, dash=dash))
fig.update_layout(title=f'Experimento 0 · {canal}: real frente a los dos modelos (sep-nov 2026)',
                  height=430, width=1100, yaxis_title='Líneas por día', xaxis_title='',
                  legend=dict(orientation='h', y=1.12))
fig.show()

Experimento 0 · Ridge, backtesting may-nov 2026 (29 cortes por canal)

                                  WAPE_%  WAPE_train_%    MAE   RMSE   R2  Sesgo_%      n
Canal           modelo                                                                   
Picking Packing Ridge honesto       23.7          22.9  173.3  215.9  0.3     -0.2  144.0
                Ridge con trampa     6.4           5.5   46.6   61.9  0.9     -0.4  144.0
TAT             Ridge honesto       23.4          16.6  340.3  448.9  0.1      1.1  143.0
                Ridge con trampa     7.3           4.8  106.4  137.1  0.9     -0.1  143.0


### Conclusiones: Experimento 0
| Canal | WAPE honesto | WAPE con trampa | Reducción aparente del error |
|---|---|---|---|
| Picking Packing | **23.7%** | 6.4% | −73% |
| TAT | **23.4%** | 7.3% | −69% |

- **Con las variables del mismo día, el error parece bajar a la cuarta parte** (de ≈24% a ≈7%) y el R² sube de 0.1-0.3 a 0.9. Sería un modelo "excelente"… que **no se puede usar**: el viernes no se sabe cuántos pedidos, SKUs o puntos de envío tendrá cada día de la semana siguiente.
- **La trampa no se detecta mirando el entrenamiento contra la validación.** El modelo con trampa tiene un error parecido en los dos (5.5% y 6.4% en PP). El data leakage no se ve como sobreajuste: se ve como un modelo "demasiado bueno". **Solo se detecta preguntando qué se sabe en el momento de predecir**, que fue la regla del notebook 02.
- **El modelo honesto ya deja una referencia:** una regresión Ridge con calendario, eventos, régimen e historia reciente se equivoca en ≈**24% del volumen** en ambos canales, sin sesgo (−0.2% PP y +1.1% TAT).
- **En TAT, el Ridge honesto muestra señales de que el futuro no se parece al pasado:** 16.6% en entrenamiento contra 23.4% en validación. Es coherente con lo que se vio en la sección 3: el nivel de TAT cambió en 2026.
- **Decisión:** las variables operativas del mismo día quedan **fuera de todos los modelos**. Las de operarios y horas se usarán **después** del pronóstico, para convertir líneas en personal (sección 11).

## 5. Líneas base

Antes de usar modelos complejos hay que saber **cuánto se logra sin modelo**. Las líneas base son pronósticos que cualquier planeador podría hacer en Excel. Si un modelo no les gana con claridad, no vale la pena su complejidad.

| Línea base | Pronóstico | Por qué |
|---|---|---|
| **Naive estacional** (`lag_7`) | Las líneas del mismo día de la semana pasada | La referencia mínima en series con patrón semanal |
| **Promedio de 4 semanas** (`prom_4_semanas`) | Promedio del mismo día de la semana en las últimas 4 semanas | Fue el **mejor predictor** del EDA (correlación 0.61 en PP); suaviza el ruido de una sola semana |

**Respaldo cuando falta el dato:** si el mismo día de la semana pasada fue festivo, `lag_7` está vacío. En ese caso se usa el promedio de 4 semanas y, si tampoco existe, el nivel reciente. Así todas las líneas base pronostican **los mismos días** y se pueden comparar.

Ninguna de las dos "se entrena": los valores ya están calculados y auditados en el notebook 02. El backtesting solo las califica en las mismas semanas que a los demás modelos.

In [10]:
# ============================================
# CELDA 5: Líneas base – naive estacional y promedio de 4 semanas
# ============================================
def respaldo(test, columnas):
    """Toma la primera columna con valor; si está vacía, la siguiente (el día de referencia fue festivo)."""
    return test[columnas].bfill(axis=1).iloc[:, 0].to_numpy()

def naive_estacional(train, test, canal):
    """Mismo día de la semana pasada. Si fue festivo: hace 2 semanas → promedio de 4 semanas → nivel reciente."""
    return respaldo(test, ['lag_7', 'lag_14', 'prom_4_semanas', 'nivel_reciente'])

def promedio_4_semanas(train, test, canal):
    """Promedio del mismo día de la semana en las últimas 4 semanas (ignora las semanas sin dato)."""
    return respaldo(test, ['prom_4_semanas', 'nivel_reciente'])

# --- ¿Cuántas veces se necesita el respaldo en la ventana de evaluación? ---
ventana = datos[datos.corte >= INICIO_EVAL]
print('Días de la ventana sin el dato principal (se usa el respaldo):')
print(ventana.groupby('Canal')[['lag_7', 'prom_4_semanas']].apply(lambda x: x.isna().sum()).to_string())

# --- Backtesting: la tabla `resultados` acumula los pronósticos de TODOS los modelos ---
resultados = pd.concat([backtest(nombre, fn, canal)
                        for canal in COL
                        for nombre, fn in [('Naive (lag_7)', naive_estacional),
                                           ('Promedio 4 semanas', promedio_4_semanas)]],
                       ignore_index=True)

print('\nLíneas base · backtesting may-nov 2026\n')
print(resumen(resultados, por=('Canal', 'modelo')).drop(columns='MAPE_%').to_string())

# --- Error por horizonte (WAPE): ¿empeora a medida que nos alejamos del corte? ---
por_h = resumen(resultados, por=('Canal', 'modelo', 'horizonte_dias'))['WAPE_%'].unstack()
print('\nWAPE % por horizonte (1 = lunes … 5 = viernes; 6 = sábado, solo hasta julio)')
print(por_h.to_string())

# --- ¿Dónde fallan? Días después de festivo frente a días normales ---
r = resultados.merge(datos[['Fecha', 'Canal', 'dia_despues_festivo']], on=['Fecha', 'Canal'])
tipo = r.assign(Tipo=np.where(r.dia_despues_festivo == 1, 'Después de festivo', 'Día normal'))
print('\nWAPE % por tipo de día (Promedio 4 semanas)')
print(resumen(tipo[tipo.modelo == 'Promedio 4 semanas'], por=('Canal', 'Tipo'))[['WAPE_%', 'Sesgo_%', 'n']].to_string())

# --- Gráfica: WAPE por horizonte ---
graf = por_h.stack().rename('WAPE_%').reset_index()
graf = graf[graf.horizonte_dias <= 5]
fig = px.line(graf, x='horizonte_dias', y='WAPE_%', color='Canal', line_dash='modelo',
              color_discrete_map=COL, markers=True,
              title='Líneas base: error por día de anticipación (may-nov 2026)')
fig.update_layout(height=420, width=900, xaxis=dict(title='Días después del corte (1 = lunes … 5 = viernes)',
                  dtick=1), yaxis_title='WAPE %', legend=dict(orientation='h', y=1.15))
fig.show()

Días de la ventana sin el dato principal (se usa el respaldo):
                 lag_7  prom_4_semanas
Canal                                 
Picking Packing      9               0
TAT                 10               0

Líneas base · backtesting may-nov 2026

                                    WAPE_%    MAE   RMSE   R2  Sesgo_%      n
Canal           modelo                                                       
Picking Packing Naive (lag_7)         31.7  231.3  289.9 -0.3     -1.5  144.0
                Promedio 4 semanas    24.3  177.8  226.9  0.2     -1.4  144.0
TAT             Naive (lag_7)         32.5  472.7  646.2 -0.9      0.9  143.0
                Promedio 4 semanas    26.5  385.3  534.9 -0.3      2.8  143.0

WAPE % por horizonte (1 = lunes … 5 = viernes; 6 = sábado, solo hasta julio)
horizonte_dias                         1     2     3     4     5     6
Canal           modelo                                                
Picking Packing Naive (lag_7)       38.3  26.4  37.6

### Conclusiones: líneas base
**La marca a superar es el promedio de 4 semanas:** WAPE de **24.3% en PP** y **26.5% en TAT**, prácticamente sin sesgo (−1.4% y +2.8%).

- **El promedio de 4 semanas le gana con claridad al naive** (24.3% contra 31.6% en PP; 26.5% contra 32.6% en TAT). Copiar una sola semana arrastra todo su ruido; promediar cuatro lo suaviza. Coincide con el EDA, donde `prom_4_semanas` fue el mejor predictor.
- **El error no crece con el horizonte: depende del día de la semana.** En este diseño, el horizonte **es** el día de la semana (1 = lunes … 5 = viernes), así que no se pueden separar los dos efectos. Lo que sí se ve:
  - **Lunes y martes son los días más difíciles** (31-42% de WAPE). En Colombia casi todos los festivos caen en lunes: los lunes y martes de las semanas anteriores quedan "contaminados" por festivos y rebotes, y el promedio de 4 semanas los arrastra. Ejemplo: el lunes 27-jul TAT tuvo 2,740 líneas y el promedio pronosticó 658.
  - **Jueves es el día más predecible** (16% en PP y 20% en TAT).
  - **Los sábados (hasta julio) eran los peores** (≈37-41%): tenían poco volumen y era muy variable.
- **Después de un festivo, TAT falla mucho:** WAPE de 62% y **sobreestima +51%**. El promedio de 4 semanas no sabe que el día después de un festivo sale bajo en TAT, como ya lo había mostrado el EDA. En PP no se nota en el agregado (24%). **Aquí es donde `dia_despues_festivo` debe aportar en los modelos con variables.**
- **Limitación detectada:** el lunes 4-may PP tuvo solo 32 líneas después del festivo del **viernes** 1 de mayo. `dia_despues_festivo` lo marca como 0, porque el día hábil anterior era el sábado 2. Es un caso raro (festivo en viernes con sábado operativo) que ya no se repite con la jornada de 42 h.
- **Primera señal de que un modelo con variables sí aporta:** el Ridge honesto del Experimento 0 (23.7% PP y 23.4% TAT) ya le gana al promedio de 4 semanas. La ventaja es pequeña en PP (−0.6 puntos) y más clara en TAT (−3.1 puntos), lo que sugiere que el calendario y el régimen ayudan más en TAT.

### 5.2 Líneas base por periodo, por mes y por día de la semana

El modelo se usará en el **régimen actual**, así que además de la ventana may-nov se revisan:
- **Dos periodos:** may-nov 2026 completo y la jornada de 42 h (desde el 15-jul, lunes a viernes), que es el más parecido a como se usará el modelo.
- **Mes a mes:** cuánto tarda la línea base en ajustarse a un cambio de régimen.
- **Por día de la semana en el régimen actual:** qué días se pronostican peor y hacia qué lado se equivocan.

In [11]:
# ============================================
# CELDA 5.2: Líneas base por periodo, por mes y por día de la semana
# ============================================
DIAS = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves', 4: 'Viernes', 5: 'Sábado'}
bases = resultados[resultados.modelo.isin(['Naive (lag_7)', 'Promedio 4 semanas'])].copy()
bases['Periodo'] = np.where(bases.Fecha >= INICIO_42H, 'Jornada 42 h (desde 15-jul)', 'May a 14-jul')

# --- 1. WAPE por periodo: toda la ventana frente al régimen actual ---
print('WAPE % por periodo\n')
print(pd.concat({'May-nov 2026': resumen(bases, por=('Canal', 'modelo'))['WAPE_%'],
                 'Jornada 42 h': resumen(bases[bases.Fecha >= INICIO_42H], por=('Canal', 'modelo'))['WAPE_%']},
                axis=1).to_string())

# --- 2. Mes a mes (Promedio 4 semanas): ¿cuánto tarda en reaccionar a un cambio de régimen? ---
p4 = bases[bases.modelo == 'Promedio 4 semanas'].assign(Mes=lambda d: d.Fecha.dt.strftime('%Y-%m'))
mes = resumen(p4, por=('Canal', 'Mes'))[['WAPE_%', 'Sesgo_%', 'n']]
mes['Real_prom'] = p4.groupby(['Canal', 'Mes']).real.mean().round(0)
mes['Pron_prom'] = p4.groupby(['Canal', 'Mes']).pred.mean().round(0)
print('\nPromedio 4 semanas, mes a mes (sesgo + = sobreestima)')
print(mes.to_string())

# --- 3. Régimen actual: WAPE y sesgo por día de la semana ---
actual = bases[bases.Fecha >= INICIO_42H].assign(Dia=lambda d: d.Fecha.dt.dayofweek.map(DIAS))
por_dia = resumen(actual, por=('Canal', 'Dia', 'modelo'))[['WAPE_%', 'Sesgo_%']].unstack('modelo')
orden = [(c, d) for c in COL for d in DIAS.values() if (c, d) in por_dia.index]
print('\nRégimen actual (jornada 42 h): WAPE y sesgo por día de la semana')
print(por_dia.loc[orden].to_string())

# --- 4. Gráfica: real frente a las líneas base (promedio semanal de líneas por día, 2026) ---
g26 = datos[datos.Fecha >= '2026-01-01'].assign(
    Naive=lambda d: respaldo(d, ['lag_7', 'lag_14', 'prom_4_semanas', 'nivel_reciente']),
    Promedio=lambda d: respaldo(d, ['prom_4_semanas', 'nivel_reciente']))
semanal = (g26.groupby(['Canal', pd.Grouper(key='Fecha', freq='W-SUN')])[[OBJETIVO, 'Naive', 'Promedio']]
              .mean().reset_index())
for canal in COL:
    d = semanal[semanal.Canal == canal]
    fig = go.Figure()
    fig.add_scatter(x=d.Fecha, y=d[OBJETIVO], name='Real', line=dict(color='#333333', width=3))
    fig.add_scatter(x=d.Fecha, y=d.Naive, name='Naive (lag_7)', line=dict(color='#9a9a9a', width=1.5, dash='dot'))
    fig.add_scatter(x=d.Fecha, y=d.Promedio, name='Promedio 4 semanas', line=dict(color=COL[canal], width=2.5))
    for fecha, texto in [('2026-05-01', 'Recorte rutas TAT'), ('2026-07-15', 'Jornada 42 h')]:
        fig.add_vline(x=pd.Timestamp(fecha), line_dash='dash', line_color='gray')
        fig.add_annotation(x=pd.Timestamp(fecha), y=1.04, yref='paper', text=texto, showarrow=False,
                           xanchor='left', font=dict(color='gray', size=11))
    fig.update_layout(title=f'{canal}: real frente a las líneas base (promedio semanal de líneas por día, 2026)',
                      height=420, width=1100, yaxis_title='Líneas por día', xaxis_title='',
                      hovermode='x unified', legend=dict(orientation='h', y=1.15))
    fig.show()

WAPE % por periodo

                                    May-nov 2026  Jornada 42 h
Canal           modelo                                        
Picking Packing Naive (lag_7)               31.7          26.7
                Promedio 4 semanas          24.3          21.3
TAT             Naive (lag_7)               32.5          25.8
                Promedio 4 semanas          26.5          20.5

Promedio 4 semanas, mes a mes (sesgo + = sobreestima)
                         WAPE_%  Sesgo_%     n  Real_prom  Pron_prom
Canal           Mes                                                 
Picking Packing 2026-05    37.2     -4.7  23.0      662.0      631.0
                2026-06    26.5     -1.4  23.0      657.0      647.0
                2026-07    22.7     -4.2  23.0      763.0      730.0
                2026-08    20.1     -5.9  19.0      893.0      840.0
                2026-09    20.2     15.2  22.0      684.0      789.0
                2026-10    23.5     -7.5  21.0      740.0      6

### Conclusiones: líneas base por periodo, mes y día
- **En el régimen actual las líneas base son aún más difíciles de superar:** el promedio de 4 semanas baja a **21.3% en PP** y **20.5% en TAT**. Con la jornada de 42 h la semana se volvió más regular. **Doble meta para los modelos:** may-nov (24.3% / 26.5%) y jornada de 42 h (21.3% / 20.5%).
- **La línea base reacciona con ≈4 semanas de retraso a los cambios de régimen** (TAT, mes a mes):
  - **Mayo:** con el recorte de rutas, el real bajó a ≈1,160 líneas por día, pero el promedio seguía en ≈1,530 → **sobreestima +31%** (WAPE 49%).
  - **Junio y julio:** pasa lo contrario. El promedio arrastra las semanas bajas de mayo mientras el volumen se recupera → **subestima −9% y −14%**.
  - Desde agosto se estabiliza (≈18-21%).

  Esta es la mayor oportunidad para los modelos con **variables de régimen**, que "avisan" del cambio desde el primer día.
- **PP, septiembre: sobreestima +15%.** El real bajó a ≈680 después de un agosto alto (≈890), y el promedio tardó en bajar. Es el mismo retraso, pero sin una decisión de negocio que lo explique.
- **Régimen actual, por día de la semana:**
  - **Lunes de PP:** WAPE de 31% y **subestima −7%**. Con la jornada de 42 h, el lunes absorbe trabajo que antes se hacía el sábado, y el promedio todavía no lo refleja del todo. **Riesgo operativo:** falta de personal los lunes.
  - **Martes de TAT:** 29% de error. Es el día después de los lunes festivos.
  - **Días más predecibles:** martes y jueves en PP (≈16%) y miércoles en TAT (15%).
  - **La única ventaja del naive** es el lunes de PP (28% contra 31%): reacciona más rápido a los cambios, aunque es más inestable el resto de la semana.
- ⚠️ **Muestras pequeñas:** 13 a 19 días por día de la semana en el régimen actual. Las diferencias entre días son orientativas.

## 6. Prophet con regresores

**Prophet** (de Meta) es un modelo de series de tiempo que descompone la demanda en piezas que se suman:

**Líneas = tendencia + patrón semanal + patrón anual + efecto de los regresores + ruido**

| Pieza | Qué captura | De dónde sale |
|---|---|---|
| Tendencia | El nivel general y sus cambios lentos | La aprende sola, con "puntos de cambio" |
| Patrón semanal | Qué días de la semana suben o bajan | Lo aprende solo a partir de la fecha |
| Patrón anual | Meses altos y bajos | Lo aprende solo a partir de la fecha |
| **Regresores** | Eventos, festivos y decisiones de negocio | **Variables del notebook 02** |

**¿Por qué encaja con este problema?** Trabaja con fechas reales, así que no le afectan los huecos de festivos ni el paso de 6 a 5 días por semana. Además, sus componentes se pueden explicar a la operación.

**Qué variables recibe y cuáles no:**
- ✅ **Regresores:** eventos (5), `dia_despues_festivo`, régimen (2 en PP y 4 en TAT) y las candidatas (`es_quincena`, `dias_operativos_semana`; `dia_mes` en TAT).
- ❌ **No recibe `dia_semana` ni `mes`:** los modela por dentro como patrón semanal y anual.
- ❌ **No recibe rezagos** (`lag_7`, `prom_4_semanas`, …): su tendencia ya sigue el nivel de la serie. Pasarle los rezagos sería darle la misma información dos veces.

**Decisión a probar: ¿aditivo o multiplicativo?**
- **Aditivo:** cada efecto suma o resta **líneas fijas** (p. ej., "el cierre de año quita 330 líneas").
- **Multiplicativo:** cada efecto sube o baja un **porcentaje** del nivel (p. ej., "el cierre de año quita 40%").

El EDA mostró efectos muy grandes en porcentaje (−73% a +64%), así que el multiplicativo podría encajar mejor. Se prueban los dos con el mismo backtesting y se elige con los datos.

In [19]:
# ============================================
# CELDA 6.1: Prophet con regresores – ¿aditivo o multiplicativo?
# ============================================
from prophet import Prophet
import logging
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').disabled = True          # oculta los mensajes de cada ajuste

# Regresores: lo que Prophet NO modela solo (la semana y el año los modela por dentro)
EVENTOS = ['semana_santa', 'semana_post_semana_santa', 'pre_cierre_anio', 'cierre_anio', 'reactivacion_anio']
REGRESORES = {
    'Picking Packing': EVENTOS + ['dia_despues_festivo', 'catalogo_2026', 'jornada_42h',
                                  'es_quincena', 'dias_operativos_semana'],
    'TAT':             EVENTOS + ['dia_despues_festivo', 'catalogo_2026', 'jornada_42h',
                                  'expansion_tat_2024', 'recorte_rutas_tat',
                                  'es_quincena', 'dias_operativos_semana', 'dia_mes'],
}

def a_prophet(df, canal):
    """Prophet exige las columnas ds (fecha) e y (objetivo) + una columna por regresor."""
    return df[['Fecha'] + REGRESORES[canal] + [OBJETIVO]].rename(columns={'Fecha': 'ds', OBJETIVO: 'y'})

def hacer_prophet(modo):
    def modelo(train, test, canal):
        m = Prophet(seasonality_mode=modo,           # aditivo: suma líneas · multiplicativo: escala en %
                    weekly_seasonality=True,          # patrón por día de la semana
                    yearly_seasonality=True,          # patrón a lo largo del año
                    daily_seasonality=False)          # datos diarios: no hay patrón dentro del día
        for r in REGRESORES[canal]:
            m.add_regressor(r)
        m.fit(a_prophet(train, canal))
        pred = m.predict(a_prophet(test, canal))['yhat'].clip(lower=0).to_numpy()
        pred_train = m.predict(a_prophet(train, canal))['yhat'].clip(lower=0).to_numpy()
        return pred, pred_train
    return modelo

# --- Paso 1: ¿aditivo o multiplicativo? Se prueban los dos con el mismo backtesting ---
partes_prophet = {}
for modo in ['additive', 'multiplicative']:
    for canal in COL:
        partes_prophet[(modo, canal)] = backtest(f'Prophet {modo}', hacer_prophet(modo), canal)

prueba_modo = pd.concat(partes_prophet.values(), ignore_index=True)
tabla_modo = resumen(prueba_modo, por=('Canal', 'modelo'))
tabla_modo['WAPE_train_%'] = [round(partes_prophet[(m.split()[1], c)].attrs['wape_train'], 1)
                              for c, m in tabla_modo.index]
tabla_modo['Segundos'] = [round(partes_prophet[(m.split()[1], c)].attrs['segundos'])
                          for c, m in tabla_modo.index]
print('Prophet · aditivo frente a multiplicativo (backtesting may-nov 2026)\n')
print(tabla_modo[['WAPE_%', 'WAPE_train_%', 'MAE', 'RMSE', 'R2', 'Sesgo_%', 'Segundos']].to_string())

Prophet · aditivo frente a multiplicativo (backtesting may-nov 2026)

                                        WAPE_%  WAPE_train_%    MAE   RMSE   R2  Sesgo_%  Segundos
Canal           modelo                                                                            
Picking Packing Prophet additive          24.1          22.6  175.9  219.1  0.3      1.5        15
TAT             Prophet additive          24.2          16.8  351.5  459.7  0.1      1.2        16
Picking Packing Prophet multiplicative    24.1          22.4  175.7  219.3  0.3      0.9        15
TAT             Prophet multiplicative    24.8          16.8  360.3  471.9  0.0      1.9        18


### Conclusiones: aditivo o multiplicativo
- **Los dos modos dan casi lo mismo en PP** (24.1% de WAPE) y **el aditivo es mejor en TAT** (24.0% contra 24.8%).
- **Se elige el modo aditivo:** es igual o mejor en los dos canales, y sus efectos se leen directo en **líneas por día**, que es como piensa la operación.
- **¿Por qué no ganó el multiplicativo si los efectos del EDA son porcentajes grandes?** El multiplicativo funciona mejor cuando el volumen crece mucho en el tiempo y los efectos crecen con él. Aquí el nivel de cada canal se mueve en un rango acotado (PP ≈650-800, TAT ≈1,400-2,000 líneas por día), así que un efecto fijo en líneas describe igual de bien los datos.
- **Sobreajuste:** el error de entrenamiento y el de validación están cerca en PP (22.6% y 24.1%). En TAT hay una brecha mayor (16.8% y 24.0%), la misma que tuvo el Ridge del Experimento 0. Más que sobreajuste, refleja que **2026 es más difícil de pronosticar en TAT** que los años anteriores (recorte de rutas y jornada de 42 h).

In [20]:
# ============================================
# CELDA 6.2: Prophet elegido (aditivo) frente a las líneas base
# ============================================
prophet_res = pd.concat([partes_prophet[('additive', c)] for c in COL], ignore_index=True)
prophet_res['modelo'] = 'Prophet'
resultados = pd.concat([resultados[resultados.modelo != 'Prophet'], prophet_res], ignore_index=True)

# --- 1. Comparación general ---
comp = resumen(resultados, por=('Canal', 'modelo'))[['WAPE_%', 'MAE', 'RMSE', 'Sesgo_%']]
print('Prophet frente a las líneas base · may-nov 2026\n')
print(comp.to_string())

# --- 2. ¿Dónde gana o pierde? Por día de la semana y después de festivo ---
r = resultados.merge(datos[['Fecha', 'Canal', 'dia_despues_festivo']], on=['Fecha', 'Canal'])
r['Tipo'] = np.where(r.dia_despues_festivo == 1, 'Después de festivo', 'Día normal')
base_vs = r[r.modelo.isin(['Promedio 4 semanas', 'Prophet'])]
print('\nWAPE % por día de la semana (1 = lunes … 5 = viernes; 6 = sábado hasta julio)')
print(resumen(base_vs, por=('Canal', 'modelo', 'horizonte_dias'))['WAPE_%'].unstack().to_string())
print('\nWAPE % y sesgo % por tipo de día')
print(resumen(base_vs, por=('Canal', 'Tipo', 'modelo'))[['WAPE_%', 'Sesgo_%', 'n']].to_string())

# --- 3. ¿Qué aprendió Prophet? Modelo entrenado con TODA la historia ---
from prophet.utilities import regressor_coefficients
modelos_prophet, efectos = {}, []
for canal in COL:
    m = Prophet(seasonality_mode='additive', daily_seasonality=False)
    for reg in REGRESORES[canal]:
        m.add_regressor(reg)
    modelos_prophet[canal] = m.fit(a_prophet(datos[datos.Canal == canal], canal))
    efectos.append(regressor_coefficients(m)[['regressor', 'coef']].assign(Canal=canal))
efectos = pd.concat(efectos).rename(columns={'regressor': 'Regresor', 'coef': 'Efecto_lineas'})
print('\nEfecto de cada regresor (líneas por día; en las numéricas, por cada unidad adicional)')
print(efectos.pivot(index='Regresor', columns='Canal', values='Efecto_lineas')
             .reindex(REGRESORES['TAT']).round(0).to_string())

# --- 4. Gráfica: efecto de los eventos y del régimen ---
mostrar = EVENTOS + ['dia_despues_festivo', 'catalogo_2026', 'jornada_42h', 'expansion_tat_2024', 'recorte_rutas_tat']
g = efectos[efectos.Regresor.isin(mostrar)]
fig = px.bar(g, x='Efecto_lineas', y='Regresor', color='Canal', barmode='group', orientation='h',
             color_discrete_map=COL, title='Lo que aprendió Prophet: efecto de cada evento y régimen (líneas por día)')
fig.add_vline(x=0, line_color='gray')
fig.update_layout(height=520, width=950, xaxis_title='Líneas por día frente a un día normal', yaxis_title='',
                  yaxis=dict(categoryorder='array', categoryarray=mostrar[::-1]),
                  legend=dict(orientation='h', y=1.1))
fig.show()

Prophet frente a las líneas base · may-nov 2026

                                    WAPE_%    MAE   RMSE  Sesgo_%
Canal           modelo                                           
Picking Packing Naive (lag_7)         31.7  231.3  289.9     -1.5
                Promedio 4 semanas    24.3  177.8  226.9     -1.4
TAT             Naive (lag_7)         32.5  472.7  646.2      0.9
                Promedio 4 semanas    26.5  385.3  534.9      2.8
Picking Packing ARIMA (regresión)     24.1  175.7  219.8      2.1
TAT             ARIMA (regresión)     26.1  379.0  492.2      2.3
Picking Packing Prophet               24.1  175.9  219.1      1.5
TAT             Prophet               24.2  351.5  459.7      1.2

WAPE % por día de la semana (1 = lunes … 5 = viernes; 6 = sábado hasta julio)
horizonte_dias                         1     2     3     4     5     6
Canal           modelo                                                
Picking Packing Promedio 4 semanas  41.5  23.0  25.5  15.9  23.4  40.7

In [9]:
# ============================================
# CELDA 6.3: ¿Cómo le va a Prophet en el régimen actual (jornada de 42 h)?
# ============================================
# --- 1. Error solo en las semanas desde el 15-jul-2026 (lunes a viernes) ---
actual = resultados[resultados.Fecha >= INICIO_42H]
print('WAPE % · régimen actual (desde el 15-jul-2026) frente a toda la ventana (may-nov)\n')
print(pd.concat({'Desde 15-jul': resumen(actual, por=('Canal', 'modelo'))['WAPE_%'],
                 'May-nov': resumen(resultados, por=('Canal', 'modelo'))['WAPE_%']}, axis=1).to_string())

# --- 2. Prueba: un patrón semanal ANTES y otro DESPUÉS de la jornada de 42 h ---
def prophet_semana_condicional(train, test, canal):
    m = Prophet(seasonality_mode='additive', daily_seasonality=False, weekly_seasonality=False)
    m.add_seasonality('semana_6_dias', period=7, fourier_order=3, condition_name='antes_42h')
    m.add_seasonality('semana_5_dias', period=7, fourier_order=3, condition_name='con_42h')
    for reg in REGRESORES[canal]:
        m.add_regressor(reg)
    def preparar(df):
        x = a_prophet(df, canal)
        x['con_42h'] = df['jornada_42h'].to_numpy() == 1
        x['antes_42h'] = ~x['con_42h']
        return x
    m.fit(preparar(train))
    return m.predict(preparar(test))['yhat'].clip(lower=0).to_numpy()

condicional = pd.concat([backtest('Prophet semana condicional', prophet_semana_condicional, c) for c in COL],
                        ignore_index=True)
print('\nPrueba: patrón semanal separado antes y después de la jornada de 42 h')
print(pd.concat({'May-nov': resumen(condicional, por=('Canal', 'modelo'))['WAPE_%'],
                 'Desde 15-jul': resumen(condicional[condicional.Fecha >= INICIO_42H],
                                         por=('Canal', 'modelo'))['WAPE_%']}, axis=1).to_string())

WAPE % · régimen actual (desde el 15-jul-2026) frente a toda la ventana (may-nov)

                                    Desde 15-jul  May-nov
Canal           modelo                                   
Picking Packing Naive (lag_7)               26.4     31.6
                Promedio 4 semanas          21.3     24.3
TAT             Naive (lag_7)               25.1     32.6
                Promedio 4 semanas          20.5     26.5
Picking Packing Prophet                     23.4     24.1
TAT             Prophet                     22.0     24.2

Prueba: patrón semanal separado antes y después de la jornada de 42 h
                                            May-nov  Desde 15-jul
Canal           modelo                                           
Picking Packing Prophet semana condicional     24.3          23.8
TAT             Prophet semana condicional     24.2          22.0


### Conclusiones: Prophet
**Resultado general:** Prophet **empata con la línea base en PP** (24.1% contra 24.3%) y **le gana en TAT** (24.0% contra 26.5%, −2.5 puntos), sin sesgo (+1.4% y +1.2%).

**Dónde gana y dónde no:**
- **TAT, martes:** baja de 35.7% a **26.2%**. Es el día después de los lunes festivos, el punto débil de la línea base.
- **TAT, después de festivo:** baja de 62% a **47%**, y el sesgo pasa de +51% a +21%. Prophet aprendió que ese día **baja ≈390 líneas**, aunque todavía no corrige todo el efecto.
- **PP, lunes:** sigue siendo el día más difícil (42%), igual que con la línea base. Ni la historia ni el calendario explican la variación de los lunes en PP.
- **PP, después de festivo:** empeora frente a la línea base (32% contra 24%) y ahora **subestima −14%**. Aprendió una caída de 142 líneas que en 2026 fue menor. Solo hay 11 días de este tipo en la ventana, así que conviene no sobreinterpretarlo.

**Lo que aprendió Prophet (efectos en líneas por día), coherente con el EDA y el notebook 02:**
- **Cierre de año:** −328 en PP y **−925 en TAT**. **Pre-cierre:** **+493 en PP** (los mayoristas anticipan pedidos).
- **Semana Santa:** ≈−340 en ambos canales; la semana siguiente recupera ≈+155.
- **Recorte de rutas TAT:** **−575 líneas por día**, el cambio de régimen más grande. **Catálogo 2026:** −135 en PP y −170 en TAT.
- **Jornada de 42 h:** +334 en TAT (el sábado se redistribuye de lunes a viernes) y casi nada en PP (+14).
- **Candidatas:** `es_quincena` resta 72 (PP) y 157 (TAT) líneas; `dia_mes` en TAT no aporta (0 por día). Se confirman o se quitan con la prueba de la sección 9.

**Nota sobre la configuración:** se dejó la flexibilidad de tendencia por defecto de Prophet (`changepoint_prior_scale = 0.05`). Una prueba rápida mostró que hacerla más flexible baja el WAPE de TAT menos de 1 punto. No se ajustó con la ventana de evaluación para **no elegir parámetros con las mismas semanas con que se califica** el modelo, porque eso haría que el error saliera optimista.

## 7. Regresión con errores ARIMA (ARIMAX)

**Idea del modelo:** es una regresión lineal (líneas = efecto del día de la semana + eventos + régimen + …) a la que se le agrega **memoria en los errores**. Si el modelo se quedó corto ayer, probablemente también se quede corto hoy; la parte ARMA aprende esa inercia.

**Decisiones de diseño:**
| Decisión | Por qué |
|---|---|
| **Sin parte estacional SARIMA** | La semana pasó de 6 a 5 días y los festivos desordenan cualquier periodo fijo. El patrón semanal va como **indicadoras de día** en la X (lunes = base). |
| **Índice diario de 7 días con huecos** | Domingos, festivos y días sin dato quedan vacíos. El filtro de Kalman los "salta" sin imputar nada, y la semana siempre mide 7 días. |
| **Sin diferenciación (d = 0)** | El EDA mostró que la serie es estacionaria una vez se descuentan los escalones de nivel, y esos escalones ya están en las variables de régimen. |
| **Sin rezagos (`lag_7`, `prom_4_semanas`…)** | La parte AR ya es la memoria del modelo; dárselos sería repetir información. |
| **Variables 0/1 con al menos 5 días** | Si en el entrenamiento una variable casi no varía (p. ej., `recorte_rutas_tat` la primera semana de mayo), su efecto no se puede estimar y se deja fuera ese corte. |

**Selección de la especificación ANTES de mirar la evaluación** (aporte del notebook del equipo):
- Se comparan **AR(1), ARMA(1,1) y AR(2)** con cortes de **enero de 2025 a abril de 2026** (uno de cada dos), sin tocar las semanas de may-nov.
- **Regla fijada de antemano:** se elige la **más simple** cuyo WAPE esté a menos de **0.5 puntos** de la mejor.
- Así se evita escoger el modelo con las mismas semanas con que se califica.

In [14]:
# ============================================
# CELDA 7.1: Regresión con errores ARIMA – preparación y selección de la especificación
# ============================================
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
NO_CONVERGIO = []                            # cortes en que el optimizador no convergió (se reportan, no se ocultan)

INICIO_AJUSTE = pd.Timestamp('2025-01-01')   # cortes para ELEGIR la especificación (antes de la evaluación)
PASO_AJUSTE   = 2                            # 1 de cada 2 semanas, para ahorrar tiempo
TOLERANCIA    = 0.5                          # se elige la más simple a menos de 0.5 puntos de la mejor
MIN_DIAS      = 5                            # una variable 0/1 entra si tiene al menos 5 días con 1 y con 0
MIN_DIAS_REGIMEN = 5                         # igual para las de régimen (en la celda 7.4 se prueba con 15)
REGIMENES = ['catalogo_2026', 'jornada_42h', 'expansion_tat_2024', 'recorte_rutas_tat']

DIAS_SEMANA = ['dow_mar', 'dow_mie', 'dow_jue', 'dow_vie', 'dow_sab']     # lunes = día base
X_ARIMA = {'Picking Packing': EVENTOS + ['dia_despues_festivo', 'catalogo_2026', 'jornada_42h',
                                         'es_quincena', 'dias_operativos_semana'],
           'TAT':             EVENTOS + ['dia_despues_festivo', 'catalogo_2026', 'jornada_42h',
                                         'expansion_tat_2024', 'recorte_rutas_tat',
                                         'es_quincena', 'dias_operativos_semana', 'dia_mes']}
NUMERICAS_ARIMA = ['dias_operativos_semana', 'dia_mes']

def exogenas(df, canal):
    """Matriz X: día de la semana como indicadoras + las variables del notebook 02."""
    X = df.set_index('Fecha')[X_ARIMA[canal]].astype(float)
    for k, nombre in enumerate(DIAS_SEMANA, start=1):
        X[nombre] = (X.index.dayofweek == k).astype(float)
    return X

def a_diario(serie_o_X, desde, hasta):
    """Índice de 7 días calendario: domingos, festivos y días sin dato quedan como NaN (el filtro de Kalman los salta)."""
    return serie_o_X.reindex(pd.date_range(desde, hasta, freq='D'))

def columnas_validas(X):
    """Quita las columnas que en el entrenamiento son constantes o casi (el modelo no podría estimar su efecto)."""
    ok = []
    for c in X.columns:
        if c in NUMERICAS_ARIMA:
            ok.append(c) if X[c].nunique() > 1 else None
        else:
            minimo = MIN_DIAS_REGIMEN if c in REGIMENES else MIN_DIAS
            if (X[c] == 1).sum() >= minimo and (X[c] == 0).sum() >= minimo:
                ok.append(c)
    return ok

def ajustar_arima(train, canal, orden):
    """Ajusta la regresión con errores ARMA sobre el índice diario. Devuelve el modelo, las columnas y la escala."""
    escala = train[OBJETIVO].mean()                                # se divide entre la media: estabilidad numérica
    X_tr = exogenas(train, canal)
    cols = columnas_validas(X_tr)
    y = a_diario(train.set_index('Fecha')[OBJETIVO] / escala, train.Fecha.min(), train.Fecha.max())
    X = a_diario(X_tr[cols], train.Fecha.min(), train.Fecha.max()).fillna(0)
    m = SARIMAX(y, exog=X, order=orden, trend='c',
                enforce_stationarity=False, enforce_invertibility=False)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')                            # los avisos se cuentan en NO_CONVERGIO
        res = m.fit(disp=False, maxiter=200)
    if not res.mle_retvals.get('converged', True):
        NO_CONVERGIO.append((canal, train.Fecha.max().date(), orden))
    return res, cols, escala

def pronosticar_arima(res, cols, escala, train, test, canal, alfa=None):
    """Pronostica desde el día siguiente al último dato hasta el último día de la semana objetivo."""
    inicio, fin = train.Fecha.max() + pd.Timedelta(days=1), test.Fecha.max()
    X_f = a_diario(exogenas(test, canal)[cols], inicio, fin).fillna(0)
    f = res.get_forecast(steps=len(X_f), exog=X_f)
    pred = (f.predicted_mean.set_axis(X_f.index).reindex(test.Fecha) * escala).clip(lower=0).to_numpy()
    if alfa is None:
        return pred
    ic = f.conf_int(alpha=alfa).set_axis(X_f.index).reindex(test.Fecha) * escala
    return pred, ic.iloc[:, 0].clip(lower=0).to_numpy(), ic.iloc[:, 1].to_numpy()

def hacer_arima(orden):
    def modelo(train, test, canal):
        res, cols, escala = ajustar_arima(train, canal, orden)
        pred = pronosticar_arima(res, cols, escala, train, test, canal)
        ajuste_train = (res.fittedvalues * escala).reindex(train.Fecha).clip(lower=0).to_numpy()
        return pred, ajuste_train
    return modelo

# --- Candidatas, de la más simple a la más compleja ---
ORDENES = {'AR(1)': (1, 0, 0), 'ARMA(1,1)': (1, 0, 1), 'AR(2)': (2, 0, 0)}

ajuste = pd.concat([backtest(n, hacer_arima(o), c, desde=INICIO_AJUSTE, hasta=INICIO_EVAL, paso=PASO_AJUSTE)
                    for n, o in ORDENES.items() for c in COL]
                   + [backtest('Promedio 4 semanas', promedio_4_semanas, c, desde=INICIO_AJUSTE,
                               hasta=INICIO_EVAL, paso=PASO_AJUSTE) for c in COL], ignore_index=True)
wape_ajuste = resumen(ajuste, por=('modelo', 'Canal'))['WAPE_%'].unstack('Canal')
n_cortes = ajuste[ajuste.modelo == 'AR(1)'].groupby('Canal').corte.nunique()
print(f'Selección con cortes de ajuste del {ajuste.corte.min().date()} al {ajuste.corte.max().date()} '
      f'({n_cortes["Picking Packing"]} en PP y {n_cortes["TAT"]} en TAT) · WAPE %\n')
print(wape_ajuste.to_string())
n_ajustes = ajuste[ajuste.modelo.isin(ORDENES)].groupby(['modelo', 'Canal']).corte.nunique().sum()
print(f'\nAjustes sin convergencia: {len(NO_CONVERGIO)} de {n_ajustes}')

ORDEN_ELEGIDO = {}
for canal in COL:
    w = wape_ajuste.loc[list(ORDENES), canal]
    nombre = next(n for n in ORDENES if w[n] <= w.min() + TOLERANCIA)       # la más simple dentro de la tolerancia
    ORDEN_ELEGIDO[canal] = ORDENES[nombre]
    print(f'\n{canal}: se elige {nombre} (WAPE de ajuste {w[nombre]:.1f}%; la mejor fue {w.min():.1f}%)')

Selección con cortes de ajuste del 2025-01-05 al 2026-04-26 (35 en PP y 34 en TAT) · WAPE %

Canal               Picking Packing   TAT
modelo                                   
AR(1)                          23.7  22.7
ARMA(1,1)                      23.6  22.5
AR(2)                          23.7  22.6
Promedio 4 semanas             29.5  22.7

Ajustes sin convergencia: 14 de 207

Picking Packing: se elige AR(1) (WAPE de ajuste 23.7%; la mejor fue 23.6%)

TAT: se elige AR(1) (WAPE de ajuste 22.7%; la mejor fue 22.5%)


### Conclusiones: selección de la especificación
- **Las tres especificaciones dan prácticamente lo mismo** (diferencias de 0.1-0.2 puntos). Por la regla de parsimonia se elige **AR(1) en los dos canales**: la memoria del modelo es solo "el error de ayer".
- **En el periodo de ajuste (2025 a abril de 2026), el ARIMA sí le gana a la línea base en PP** (23.7% contra 29.5%) y empata en TAT (22.7%). Ese periodo incluye un cierre de año y una Semana Santa, donde las variables de eventos aportan mucho.
- **13 de 207 ajustes no convergieron del todo.** Se reportan, y su pronóstico se usa igual; son pocos y no cambian la elección.

In [15]:
# ============================================
# CELDA 7.2: Regresión con errores ARIMA – evaluación en may-nov 2026
# ============================================
INTERVALOS = []                                  # guarda el intervalo del 80 % de cada pronóstico
NO_CONVERGIO.clear()

def arima_final(train, test, canal):
    res, cols, escala = ajustar_arima(train, canal, ORDEN_ELEGIDO[canal])
    pred, lo, hi = pronosticar_arima(res, cols, escala, train, test, canal, alfa=0.2)
    INTERVALOS.append(pd.DataFrame({'Fecha': test.Fecha.to_numpy(), 'Canal': canal, 'lo80': lo, 'hi80': hi}))
    ajuste_train = (res.fittedvalues * escala).reindex(train.Fecha).clip(lower=0).to_numpy()
    return pred, ajuste_train

partes_arima = [backtest('ARIMA (regresión)', arima_final, c) for c in COL]
arima_res = pd.concat(partes_arima, ignore_index=True)
resultados = pd.concat([resultados[resultados.modelo != 'ARIMA (regresión)'], arima_res], ignore_index=True)

# --- 1. Comparación con los modelos anteriores, en las dos ventanas ---
comp = pd.concat({'WAPE may-nov': resumen(resultados, por=('Canal', 'modelo'))['WAPE_%'],
                  'WAPE 42 h':    resumen(resultados[resultados.Fecha >= INICIO_42H], por=('Canal', 'modelo'))['WAPE_%'],
                  'Sesgo may-nov': resumen(resultados, por=('Canal', 'modelo'))['Sesgo_%']}, axis=1)
print('Regresión con errores ARIMA frente a los modelos anteriores\n')
print(comp.sort_index(level='Canal', sort_remaining=False).to_string())
for canal, p in zip(COL, partes_arima):
    print(f'  {canal:16s}: WAPE de entrenamiento {p.attrs["wape_train"]:.1f}% · {p.attrs["segundos"]:.0f} s')
print(f'  Ajustes sin convergencia: {len(NO_CONVERGIO)} de 58 → {NO_CONVERGIO}')

# --- 2. ¿Dónde gana o pierde? Después de festivo ---
r = resultados.merge(datos[['Fecha', 'Canal', 'dia_despues_festivo']], on=['Fecha', 'Canal'])
r['Tipo'] = np.where(r.dia_despues_festivo == 1, 'Después de festivo', 'Día normal')
print('\nWAPE % y sesgo % después de festivo')
print(resumen(r[r.Tipo == 'Después de festivo'], por=('Canal', 'modelo'))[['WAPE_%', 'Sesgo_%', 'n']].to_string())

# --- 3. Intervalos del 80 %: ¿contienen el valor real ≈80 % de las veces? ---
ic = arima_res.merge(pd.concat(INTERVALOS), on=['Fecha', 'Canal'])
ic['dentro'] = ic.real.between(ic.lo80, ic.hi80)
print('\nCobertura del intervalo del 80 % (ideal ≈ 80 %):')
print((ic.groupby('Canal').dentro.mean() * 100).round(1).to_string())

Regresión con errores ARIMA frente a los modelos anteriores

                                    WAPE may-nov  WAPE 42 h  Sesgo may-nov
Canal           modelo                                                    
Picking Packing Naive (lag_7)               31.7       26.7           -1.5
                Promedio 4 semanas          24.3       21.3           -1.4
                ARIMA (regresión)           24.1       23.3            2.1
TAT             Naive (lag_7)               32.5       25.8            0.9
                Promedio 4 semanas          26.5       20.5            2.8
                ARIMA (regresión)           26.1       23.1            2.3
  Picking Packing : WAPE de entrenamiento 23.1% · 251 s
  TAT             : WAPE de entrenamiento 17.3% · 514 s
  Ajustes sin convergencia: 1 de 58 → [('TAT', datetime.date(2026, 7, 17), (1, 0, 0))]

WAPE % y sesgo % después de festivo
                                    WAPE_%  Sesgo_%     n
Canal           modelo                       

In [16]:
# ============================================
# CELDA 7.3: Regresión con errores ARIMA – qué aprendió y diagnóstico de residuales
# ============================================
for canal in COL:
    d = datos[datos.Canal == canal]
    res, cols, escala = ajustar_arima(d, canal, ORDEN_ELEGIDO[canal])      # toda la historia
    coef = pd.DataFrame({'Efecto_lineas': res.params, 'p_valor': res.pvalues})
    en_lineas = coef.index.isin(['intercept'] + cols)
    coef.loc[en_lineas, 'Efecto_lineas'] *= escala                         # se deshace la escala
    nombre = next(n for n, o in ORDENES.items() if o == ORDEN_ELEGIDO[canal])
    print(f'=== {canal} · {nombre} · entrenado con toda la historia ===')
    print(coef.round({'Efecto_lineas': 2, 'p_valor': 3}).to_string())

    resid = pd.Series(res.resid).reindex(d.Fecha).dropna().iloc[10:]       # solo días con dato
    lb = acorr_ljungbox(resid, lags=[5, 10, 20], model_df=sum(ORDEN_ELEGIDO[canal][::2]), return_df=True)
    print('\nLjung-Box sobre los residuales (p < 0.05 = queda patrón sin explicar):')
    print(lb.round(3).to_string(), '\n')

=== Picking Packing · AR(1) · entrenado con toda la historia ===
                          Efecto_lineas  p_valor
intercept                        560.47    0.000
semana_santa                    -329.57    0.000
semana_post_semana_santa         166.52    0.003
pre_cierre_anio                  511.24    0.000
cierre_anio                     -477.53    0.000
reactivacion_anio                 34.20    0.386
dia_despues_festivo             -133.66    0.000
catalogo_2026                    -55.23    0.011
jornada_42h                       17.44    0.624
es_quincena                      -61.17    0.000
dias_operativos_semana           -26.69    0.139
dow_mar                          222.96    0.000
dow_mie                          370.53    0.000
dow_jue                          521.62    0.000
dow_vie                          289.06    0.000
dow_sab                         -126.86    0.000
ar.L1                              0.15    0.000
sigma2                             0.09    0.000

Lju

In [17]:
# ============================================
# CELDA 7.4: Sensibilidad – ¿y si un régimen solo se usa cuando ya tiene 15 días de datos?
# ============================================
MIN_DIAS_REGIMEN = 15                                         # regla del notebook ARIMAX del equipo
sensib = pd.concat([backtest('ARIMA · régimen con 15 días', arima_final, c) for c in COL], ignore_index=True)
MIN_DIAS_REGIMEN = 5                                          # se restaura la regla del proyecto

compara = pd.concat([arima_res, sensib], ignore_index=True)
print('WAPE % y sesgo %: régimen desde el día 5 (proyecto) frente a régimen desde el día 15\n')
print(resumen(compara, por=('Canal', 'modelo'))[['WAPE_%', 'Sesgo_%']].to_string())
mayo_jun = compara[compara.Fecha < '2026-07-01'].assign(Mes=lambda d: d.Fecha.dt.strftime('%Y-%m'))
print('\nMayo y junio (los meses del recorte de rutas TAT):')
print(resumen(mayo_jun, por=('Canal', 'Mes', 'modelo'))[['WAPE_%', 'Sesgo_%']].to_string())

WAPE % y sesgo %: régimen desde el día 5 (proyecto) frente a régimen desde el día 15

                                             WAPE_%  Sesgo_%
Canal           modelo                                      
Picking Packing ARIMA (regresión)              24.1      2.1
TAT             ARIMA (regresión)              26.1      2.3
Picking Packing ARIMA · régimen con 15 días    24.1      1.8
TAT             ARIMA · régimen con 15 días    28.4      4.5

Mayo y junio (los meses del recorte de rutas TAT):
                                                     WAPE_%  Sesgo_%
Canal           Mes     modelo                                      
Picking Packing 2026-05 ARIMA (regresión)              32.6     -1.8
                2026-06 ARIMA (regresión)              22.8     -0.9
TAT             2026-05 ARIMA (regresión)              38.7      4.9
                2026-06 ARIMA (regresión)              30.6    -16.0
Picking Packing 2026-05 ARIMA · régimen con 15 días    32.6     -1.8
             

### Conclusiones: regresión con errores ARIMA
**Resultado general:** el ARIMA **empata con Prophet en PP** (24.1%) y queda **casi igual a la línea base en TAT** (26.1% contra 26.5%), sin sesgo importante (+2.0% y +2.3%). **En el régimen actual pierde contra el promedio de 4 semanas** en los dos canales (23.3% contra 21.3% en PP; 23.1% contra 20.5% en TAT), igual que Prophet.

**Lo que aprendió (efectos en líneas por día, coherentes con Prophet y con el EDA):**
- **Patrón semanal de PP:** frente al lunes, el jueves suma **+521** líneas y el miércoles +371. Es el día de la semana lo que más explica PP.
- **Eventos:** cierre de año **−477 PP / −790 TAT**; pre-cierre **+511 en PP**; Semana Santa ≈ −325 en ambos canales.
- **Régimen TAT:** recorte de rutas **−547**, expansión 2024 +183, jornada de 42 h +260.
- **Después de festivo:** −134 PP / **−388 TAT**. Es casi el mismo valor que aprendió Prophet (−142 / −390): dos modelos distintos llegan al mismo efecto, lo que le da solidez al hallazgo.
- **Candidatas:** `es_quincena` sí es significativa (−61 PP, −143 TAT; p < 0.001). `dias_operativos_semana` (p = 0.14 y 0.80) y `dia_mes` (p = 0.70) **no lo son**. Se confirma en la sección 9.

**La memoria (AR) casi no aporta:**
- En PP, `ar.L1` = 0.15: el error de ayer explica muy poco del de hoy.
- **En TAT, `ar.L1` = 0.02 (p = 0.46): no hay memoria.** Coincide con el EDA: en TAT, lo que pasó ayer no ayuda a predecir hoy.
- En la práctica, **este modelo es una regresión lineal de calendario y régimen**. Por eso se parece tanto al Ridge del Experimento 0 (23.7% PP, 23.4% TAT).

**Diagnóstico:**
- **Ljung-Box:** sin patrón en los errores a 5 y 10 días (p > 0.08), pero **sí a 20 días** (p < 0.05). Queda una estructura de varias semanas que el modelo no capta; puede ser el nivel que cambia mes a mes.
- **Intervalos del 80% bien calibrados:** contienen el real el **78.5%** de las veces en PP y el **82.5%** en TAT. Es una ventaja práctica: el ARIMA puede dar un **rango** para planear personal, no solo un número.
- **Sobreajuste:** no hay en PP (23.1% en entrenamiento contra 24.1% en validación). En TAT aparece la misma brecha de los otros modelos (17.3% contra 26.1%), causada por los cambios de 2026.

**Sensibilidad a la regla de los regímenes**:
- Si un régimen solo se usa después de 15 días de datos, **TAT empeora de 26.1% a 28.4%**, y **mayo pasa de 38.7% a 57.6%** de error, con sesgo de **+39%**: durante las primeras semanas del recorte de rutas, el modelo no sabe que ocurrió.
- **Implicación para la operación:** cuando haya una decisión de negocio que cambie el volumen (recorte de rutas, nuevo catálogo), **conviene avisarle al modelo desde el primer día**. Esperar a que los datos la reflejen cuesta ≈19 puntos de error en el primer mes.

## 8. Gradient Boosting con rezagos

**Sus riesgos con estos datos:**
- **No puede extrapolar:** un árbol solo predice valores dentro de lo que ya vio. Si el nivel de TAT baja a un valor nuevo, el árbol tarda en "creerlo".
- **Sobreajuste:** con pocos días por evento y 87 días del régimen actual, puede memorizar el ruido. Por eso se revisa el error de entrenamiento frente al de validación.

**Implementación:** `HistGradientBoostingRegressor` de scikit-learn, con `dia_semana` y `mes` como **categorías**.

**Selección de la configuración antes de mirar la evaluación** (misma regla que el ARIMA): se comparan 4 configuraciones en los cortes de ajuste (ene-2025 a abr-2026) y se elige la más simple a menos de 0.5 puntos de la mejor:
| Configuración | Profundidad de cada árbol | Qué aprende |
|---|---|---|
| Poco profundo | 3 niveles | Las líneas directamente |
| Poco profundo · relativo | 3 niveles | La **proporción** líneas ÷ nivel reciente (para ayudarle con los cambios de nivel) |
| Profundo | 6 niveles | Las líneas directamente |
| Profundo · relativo | 6 niveles | La proporción |

Todas usan 300 árboles, tasa de aprendizaje de 0.05, un mínimo de 20 días por hoja y regularización L2. Son valores conservadores para no sobreajustar.

In [23]:
# ============================================
# CELDA 8.1: Gradient Boosting – preparación y selección de la configuración
# ============================================
from sklearn.ensemble import HistGradientBoostingRegressor

CATEGORICAS_GB = ['dia_semana', 'mes']        # el árbol las trata como categorías, no como números

def hacer_gb(params, objetivo='lineas'):
    """Gradient Boosting con las variables del notebook 02 (rezagos + calendario + eventos + régimen).

    objetivo='lineas'  : aprende las líneas directamente.
    objetivo='relativo': aprende líneas ÷ nivel_reciente y luego multiplica. Así el árbol aprende
                         PROPORCIONES (p. ej., 'el jueves es 1.3 veces el nivel') que siguen valiendo
                         cuando el nivel cambia, algo que un árbol solo no puede extrapolar."""
    def modelo(train, test, canal):
        cols = FEATURES[canal]
        tr = train.dropna(subset=['nivel_reciente'])                     # las primeras semanas no tienen historia
        y = tr[OBJETIVO] / tr['nivel_reciente'] if objetivo == 'relativo' else tr[OBJETIVO]
        m = HistGradientBoostingRegressor(categorical_features=[cols.index(c) for c in CATEGORICAS_GB],
                                          random_state=42, **params)
        m.fit(tr[cols], y)
        escala_te = test['nivel_reciente'].to_numpy() if objetivo == 'relativo' else 1
        escala_tr = train['nivel_reciente'].to_numpy() if objetivo == 'relativo' else 1
        pred = np.clip(m.predict(test[cols]) * escala_te, 0, None)
        pred_train = np.clip(m.predict(train[cols]) * escala_tr, 0, None)
        return pred, pred_train
    return modelo

# --- Candidatas: de la más simple a la más compleja ---
BASE = dict(learning_rate=0.05, max_iter=300, min_samples_leaf=20, l2_regularization=1.0)
CANDIDATAS_GB = {
    'GB poco profundo':            (dict(BASE, max_depth=3), 'lineas'),
    'GB poco profundo · relativo': (dict(BASE, max_depth=3), 'relativo'),
    'GB profundo':                 (dict(BASE, max_depth=6), 'lineas'),
    'GB profundo · relativo':      (dict(BASE, max_depth=6), 'relativo'),
}

# --- Selección con los mismos cortes de ajuste del ARIMA (ene-2025 a abr-2026, 1 de cada 2) ---
ajuste_gb = pd.concat([backtest(n, hacer_gb(p, o), c, desde=INICIO_AJUSTE, hasta=INICIO_EVAL, paso=PASO_AJUSTE)
                       for n, (p, o) in CANDIDATAS_GB.items() for c in COL], ignore_index=True)
ajuste_gb = pd.concat([ajuste_gb, ajuste[ajuste.modelo == 'Promedio 4 semanas']], ignore_index=True)
tabla_ajuste_gb = resumen(ajuste_gb, por=('modelo', 'Canal'))[['WAPE_%', 'Sesgo_%']].unstack('Canal')
print('Selección con los cortes de ajuste (ene-2025 a abr-2026) · WAPE % y sesgo %\n')
print(tabla_ajuste_gb.to_string())

CONFIG_GB = {}
for canal in COL:
    w = tabla_ajuste_gb[('WAPE_%', canal)].loc[list(CANDIDATAS_GB)]
    nombre = next(n for n in CANDIDATAS_GB if w[n] <= w.min() + TOLERANCIA)    # misma regla de parsimonia
    CONFIG_GB[canal] = nombre
    print(f'\n{canal}: se elige "{nombre}" (WAPE de ajuste {w[nombre]:.1f}%; la mejor fue {w.min():.1f}%)')

Selección con los cortes de ajuste (ene-2025 a abr-2026) · WAPE % y sesgo %

                                     WAPE_%               Sesgo_%     
Canal                       Picking Packing   TAT Picking Packing  TAT
modelo                                                                
GB poco profundo                       23.6  19.8             3.3  4.5
GB poco profundo · relativo            28.4  25.0             2.5  0.2
GB profundo                            23.5  19.6             2.4  4.7
GB profundo · relativo                 26.6  26.3             2.0  1.3
Promedio 4 semanas                     29.5  22.7             6.2  3.5

Picking Packing: se elige "GB poco profundo" (WAPE de ajuste 23.6%; la mejor fue 23.5%)

TAT: se elige "GB poco profundo" (WAPE de ajuste 19.8%; la mejor fue 19.6%)


### Conclusiones: selección de la configuración
- **Se elige el GB poco profundo (3 niveles) en los dos canales.** El profundo es apenas mejor (23.5% contra 23.6% en PP; 19.6% contra 19.8% en TAT), pero la diferencia está dentro de la tolerancia de 0.5 puntos, así que gana el más simple. Que los árboles profundos casi no mejoren sugiere que **no hay interacciones complejas** que aprovechar, y el modelo simple tiene menos riesgo de sobreajuste.
- **La versión relativa empeora** (+3 a +6 puntos). Al dividir entre el nivel reciente, cualquier día raro en las dos semanas anteriores al corte distorsiona todos los pronósticos de la semana siguiente. La idea no funcionó y se descarta con evidencia.
- **En el periodo de ajuste, el GB ya le gana a la línea base en los dos canales** (23.6% contra 29.5% en PP; 19.8% contra 22.7% en TAT). Es el mejor resultado de ajuste hasta ahora, por delante del ARIMA.

In [24]:
# ============================================
# CELDA 8.2: Gradient Boosting – evaluación en may-nov 2026
# ============================================
partes_gb = [backtest('Gradient Boosting', hacer_gb(*CANDIDATAS_GB[CONFIG_GB[c]]), c) for c in COL]
gb_res = pd.concat(partes_gb, ignore_index=True)
resultados = pd.concat([resultados[resultados.modelo != 'Gradient Boosting'], gb_res], ignore_index=True)

# --- 1. Comparación con todos los modelos, en las dos ventanas ---
comp = pd.concat({'WAPE may-nov': resumen(resultados, por=('Canal', 'modelo'))['WAPE_%'],
                  'WAPE 42 h':    resumen(resultados[resultados.Fecha >= INICIO_42H], por=('Canal', 'modelo'))['WAPE_%'],
                  'Sesgo may-nov': resumen(resultados, por=('Canal', 'modelo'))['Sesgo_%']}, axis=1)
print('Gradient Boosting frente a los modelos anteriores\n')
print(comp.sort_index(level='Canal', sort_remaining=False).to_string())

# --- 2. Sobreajuste (sesión 5): error en entrenamiento frente a validación ---
print('\nSobreajuste: WAPE % en entrenamiento frente a validación (may-nov)')
for canal, p in zip(COL, partes_gb):
    val = metricas(p.real, p.pred)['WAPE_%']
    print(f'  {canal:16s}: entrenamiento {p.attrs["wape_train"]:.1f}% · validación {val:.1f}% · '
          f'brecha {val - p.attrs["wape_train"]:+.1f} puntos · {p.attrs["segundos"]:.0f} s')

# --- 3. Después de festivo ---
r = resultados.merge(datos[['Fecha', 'Canal', 'dia_despues_festivo']], on=['Fecha', 'Canal'])
print('\nWAPE % y sesgo % después de festivo')
print(resumen(r[r.dia_despues_festivo == 1], por=('Canal', 'modelo'))[['WAPE_%', 'Sesgo_%', 'n']]
      .sort_index(level='Canal', sort_remaining=False).to_string())

Gradient Boosting frente a los modelos anteriores

                                    WAPE may-nov  WAPE 42 h  Sesgo may-nov
Canal           modelo                                                    
Picking Packing Naive (lag_7)               31.7       26.7           -1.5
                Promedio 4 semanas          24.3       21.3           -1.4
                ARIMA (regresión)           24.1       23.3            2.1
                Prophet                     24.1       23.4            1.5
                Gradient Boosting           23.6       21.7           -2.1
TAT             Naive (lag_7)               32.5       25.8            0.9
                Promedio 4 semanas          26.5       20.5            2.8
                ARIMA (regresión)           26.1       23.1            2.3
                Prophet                     24.2       22.0            1.2
                Gradient Boosting           26.2       18.5           -1.7

Sobreajuste: WAPE % en entrenamiento frente a va

In [25]:
# ============================================
# CELDA 8.3: Gradient Boosting – ¿cuándo gana y qué variables usa?
# ============================================
from sklearn.inspection import permutation_importance

# --- 1. Mes a mes frente a la línea base ---
mm = resultados[resultados.modelo.isin(['Promedio 4 semanas', 'Gradient Boosting'])]
mm = mm.assign(Mes=mm.Fecha.dt.strftime('%Y-%m'))
print('WAPE % mes a mes: Promedio 4 semanas frente a Gradient Boosting\n')
print(resumen(mm, por=('Canal', 'Mes', 'modelo'))['WAPE_%'].unstack('modelo').to_string())

# --- 2. Importancia por permutación: ¿cuánto empeora el error si se "desordena" cada variable? ---
#     Modelo entrenado hasta el inicio de la evaluación y medido en may-nov (datos que no vio).
importancias = []
for canal in COL:
    cols = FEATURES[canal]
    params, _ = CANDIDATAS_GB[CONFIG_GB[canal]]
    d = datos[datos.Canal == canal]
    tr = d[d.Fecha < INICIO_EVAL].dropna(subset=['nivel_reciente'])
    ev = d[d.corte >= INICIO_EVAL]
    m = HistGradientBoostingRegressor(categorical_features=[cols.index(c) for c in CATEGORICAS_GB],
                                      random_state=42, **params).fit(tr[cols], tr[OBJETIVO])
    pi = permutation_importance(m, ev[cols], ev[OBJETIVO], scoring='neg_mean_absolute_error',
                                n_repeats=20, random_state=42)
    importancias.append(pd.DataFrame({'Variable': cols, 'Canal': canal,
                                      'Aumento_MAE': pi.importances_mean}))
importancias = pd.concat(importancias)
top = importancias.pivot(index='Variable', columns='Canal', values='Aumento_MAE')
top = top.loc[top.max(axis=1).sort_values(ascending=False).index]
print('\nImportancia por permutación (líneas de error que se AGREGAN al desordenar la variable)')
print(top.round(1).to_string())

# --- 3. Gráfica de importancia ---
g = importancias[importancias.Variable.isin(top.index[:12])]
fig = px.bar(g, x='Aumento_MAE', y='Variable', color='Canal', barmode='group', orientation='h',
             color_discrete_map=COL, title='Gradient Boosting: variables que más aportan (importancia por permutación)')
fig.update_layout(height=520, width=950, xaxis_title='Líneas de error que se agregan al desordenar la variable',
                  yaxis_title='', yaxis=dict(categoryorder='array', categoryarray=list(top.index[:12])[::-1]),
                  legend=dict(orientation='h', y=1.1))
fig.show()

WAPE % mes a mes: Promedio 4 semanas frente a Gradient Boosting

modelo                   Promedio 4 semanas  Gradient Boosting
Canal           Mes                                           
Picking Packing 2026-05                37.2               33.5
                2026-06                26.5               26.1
                2026-07                22.7               17.9
                2026-08                20.1               22.4
                2026-09                20.2               18.7
                2026-10                23.5               24.7
                2026-11                19.1               22.4
TAT             2026-05                48.6               54.3
                2026-06                32.8               39.0
                2026-07                27.7               26.7
                2026-08                19.8               16.8
                2026-09                17.7               16.4
                2026-10                18.9          

### Conclusiones: Gradient Boosting
**Resultado general:**
| | PP may-nov | PP jornada 42 h | TAT may-nov | TAT jornada 42 h |
|---|---|---|---|---|
| Promedio 4 semanas | 24.3% | **21.3%** | 26.5% | 20.5% |
| Prophet | 24.1% | 23.4% | **24.2%** | 22.0% |
| **Gradient Boosting** | **23.6%** | 21.7% | 26.2% | **18.5%** |

- **PP:** es el **mejor modelo en may-nov** (23.6%) y **empata con la línea base en el régimen actual** (21.7% contra 21.3%).
- **TAT:** es **el único modelo que le gana a la línea base en el régimen actual: 18.5% contra 20.5%** (−2.0 puntos). Desde agosto le gana todos los meses; en octubre baja a **12.9%** contra 18.9%. En may-nov completo queda casi igual a la línea base (26.2% contra 26.5%) por lo que pasa en mayo y junio.
- **Sesgo levemente negativo** (−2.1% PP, −1.7% TAT): subestima un poco. Es pequeño, pero va en la dirección que más le cuesta a la operación (falta de personal) y hay que vigilarlo.

**Cuándo falla: los cambios de régimen.** En **mayo y junio de TAT** es el peor modelo (54.3% y 39.0%, contra 48.6% y 32.8% de la línea base). Es su límite conocido: **un árbol no puede extrapolar**. Cuando llegó el recorte de rutas, el nivel de TAT bajó a valores que el árbol no había visto con esas combinaciones de variables, y tardó varias semanas en ajustarse. Cuando el régimen se estabiliza (agosto en adelante), aprovecha mejor que nadie la combinación de historia reciente y calendario.

**En PP los meses se reparten:** el GB gana en mayo, julio y septiembre (julio: 17.9% contra 22.7%), y la línea base en agosto, octubre y noviembre. Las diferencias son de 1 a 3 puntos, coherentes con el empate en el régimen actual.

**Qué variables usa (importancia por permutación, may-nov):**
- **`dia_semana` es la más importante en los dos canales:** desordenarla agrega **47 líneas de error en PP** y 30 en TAT.
- **En TAT pesan los rezagos** (`lag_28` +19, `lag_7` +16), **`mes`** y **`dia_despues_festivo`** (+8 cada una). En PP, casi solo el día de la semana: los rezagos individuales aportan poco (≤4 líneas).
- **Los eventos y el régimen marcan 0, y es lo esperado:** may-nov no tiene Semana Santa ni cierre de año, y este modelo de prueba se entrenó hasta abril, cuando `recorte_rutas_tat` y `jornada_42h` todavía no existían. En el backtesting el modelo se reentrena cada semana y los va aprendiendo; aquí no se pueden medir.
- **`prom_4_semanas` no aporta** (−5 líneas en TAT: desordenarla incluso ayuda un poco): el árbol ya obtiene esa información de los rezagos sueltos. Tampoco aportan `dias_operativos_semana` ni `n_lags_disponibles`.

**Sobreajuste:**
- **PP: brecha moderada** (17.8% en entrenamiento contra 23.6% en validación, +5.8 puntos).
- **TAT: brecha grande** (12.1% contra 26.2%, **+14.1 puntos**), la mayor de todos los modelos (Prophet ≈+7, ARIMA ≈+9). Hay **algo de sobreajuste** sumado al cambio de régimen de 2026. Por eso se eligió la configuración más conservadora (3 niveles y regularización): los árboles profundos memorizan más.

**Después de festivo:** no mejora a Prophet (TAT 59.1% contra 47.1%). Usa `dia_despues_festivo`, pero con solo ≈60 días de ese tipo en 4 años, no termina de aprender el efecto completo.

> **Nota de reproducibilidad:** con otras versiones de scikit-learn, el WAPE puede variar unas décimas; las conclusiones no cambian.

## 9. Prueba de las variables candidatas

En el notebook 02 quedaron tres variables como **candidatas débiles**: entraron al modelo, pero su aporte no era claro.
| Variable | Canal | Evidencia del notebook 02 |
|---|---|---|
| `es_quincena` | Ambos | −4%, p ≈ 0.02 (posible falso positivo) |
| `dias_operativos_semana` | Ambos | ρ = −0.22 en PP; débil |
| `dia_mes` | TAT | Patrón nuevo de 2026 (ρ = +0.54) |

**Método (ablación):** se entrena el Gradient Boosting elegido **quitando una candidata a la vez**, y también **sin ninguna**, y se mide cuánto cambia el WAPE. Si quitar una variable no empeora el error, la variable no aporta.

**Se prueba en el Gradient Boosting** porque es el mejor modelo en el régimen actual y se reentrena en segundos. En Prophet y ARIMA ya hay evidencia complementaria (sus coeficientes y p-valores, secciones 6 y 7).

**Regla de decisión, fijada antes de ver los resultados:**
- Se mide el cambio en **tres ventanas**: ajuste (ene-2025 a abr-2026), may-nov 2026 y jornada de 42 h.
- **Una candidata se quita si quitarla no empeora el WAPE más de 0.25 puntos en ninguna de las tres.** Diferencias menores a 0.25 puntos se consideran ruido.
- Así se exige que una variable demuestre su aporte en todos los periodos, no solo en uno. Ante la duda, gana el modelo más simple.

In [28]:
# ============================================
# CELDA 9: Prueba de las variables candidatas (Gradient Boosting)
# ============================================
def gb_sin(quitar):
    """Gradient Boosting elegido en la sección 8, pero SIN las variables de `quitar`."""
    def modelo(train, test, canal):
        cols = [c for c in FEATURES[canal] if c not in quitar]
        params, _ = CANDIDATAS_GB[CONFIG_GB[canal]]
        tr = train.dropna(subset=['nivel_reciente'])
        m = HistGradientBoostingRegressor(categorical_features=[cols.index(c) for c in CATEGORICAS_GB],
                                          random_state=42, **params).fit(tr[cols], tr[OBJETIVO])
        return np.clip(m.predict(test[cols]), 0, None)
    return modelo

PRUEBAS = {'Con todas':                  [],
           'Sin es_quincena':            ['es_quincena'],
           'Sin dias_operativos_semana': ['dias_operativos_semana'],
           'Sin dia_mes':                ['dia_mes'],
           'Sin ninguna candidata':      ['es_quincena', 'dias_operativos_semana', 'dia_mes']}

filas = []
for canal in COL:
    for nombre, quitar in PRUEBAS.items():
        if nombre == 'Sin dia_mes' and canal == 'Picking Packing':
            continue                                                  # dia_mes solo es candidata en TAT
        aj = backtest(nombre, gb_sin(quitar), canal, desde=INICIO_AJUSTE, hasta=INICIO_EVAL, paso=PASO_AJUSTE)
        ev = backtest(nombre, gb_sin(quitar), canal)
        filas.append({'Canal': canal, 'Prueba': nombre,
                      'WAPE ajuste': metricas(aj.real, aj.pred)['WAPE_%'],
                      'WAPE may-nov': metricas(ev.real, ev.pred)['WAPE_%'],
                      'WAPE 42 h': metricas(ev[ev.Fecha >= INICIO_42H].real, ev[ev.Fecha >= INICIO_42H].pred)['WAPE_%']})
tabla_cand = pd.DataFrame(filas).set_index(['Canal', 'Prueba'])

# Cambio frente a "Con todas": negativo = quitar la variable MEJORA el error
cambio = tabla_cand - tabla_cand.xs('Con todas', level='Prueba').reindex(tabla_cand.index, level='Canal')
print('WAPE % con y sin cada candidata (GB elegido)\n')
print(tabla_cand.round(1).to_string())
print('\nCambio en puntos frente a "Con todas" (negativo = quitarla MEJORA; positivo = la variable aporta)\n')
print(cambio.drop(index='Con todas', level='Prueba').round(2).to_string())

WAPE % con y sin cada candidata (GB elegido)

                                            WAPE ajuste  WAPE may-nov  WAPE 42 h
Canal           Prueba                                                          
Picking Packing Con todas                          23.6          23.6       21.7
                Sin es_quincena                    23.7          24.2       22.3
                Sin dias_operativos_semana         23.4          23.3       21.4
                Sin ninguna candidata              23.7          24.0       22.1
TAT             Con todas                          19.8          26.2       18.5
                Sin es_quincena                    19.8          26.1       18.3
                Sin dias_operativos_semana         19.3          25.9       18.5
                Sin dia_mes                        19.9          27.1       20.8
                Sin ninguna candidata              19.2          27.0       20.8

Cambio en puntos frente a "Con todas" (negativo = quitarla MEJ

In [29]:
# ============================================
# CELDA 9.2: Decisión y Gradient Boosting final con las variables depuradas
# ============================================
UMBRAL = 0.25    # puntos de WAPE: por debajo de esto, la diferencia se considera ruido

QUITAR = {}
for canal in COL:
    QUITAR[canal] = []
    for v in CANDIDATAS[canal]:
        aporte = cambio.loc[(canal, f'Sin {v}')]                     # cuánto empeora el error al quitarla
        decision = 'se quita' if (aporte <= UMBRAL).all() else 'se queda'
        if decision == 'se quita':
            QUITAR[canal].append(v)
        print(f'{canal:16s} · {v:24s}: aporte máximo {aporte.max():+.2f} puntos → {decision}')

# --- Variables finales del GB y su evaluación ---
FEATURES_GB = {c: [v for v in FEATURES[c] if v not in QUITAR[c]] for c in COL}
print({c: len(v) for c, v in FEATURES_GB.items()}, 'variables finales')

partes = [backtest('Gradient Boosting', gb_sin(QUITAR[c]), c) for c in COL]
gb_final = pd.concat(partes, ignore_index=True)
aj_final = pd.concat([backtest('GB final', gb_sin(QUITAR[c]), c, desde=INICIO_AJUSTE, hasta=INICIO_EVAL,
                               paso=PASO_AJUSTE) for c in COL], ignore_index=True)
antes = resultados[resultados.modelo == 'Gradient Boosting']
print('\nGradient Boosting: con todas las candidatas frente a las variables depuradas')
print(pd.DataFrame({
    'Ajuste (depurado)':  resumen(aj_final, por=('Canal',))['WAPE_%'],
    'May-nov antes':      resumen(antes, por=('Canal',))['WAPE_%'],
    'May-nov depurado':   resumen(gb_final, por=('Canal',))['WAPE_%'],
    '42 h antes':         resumen(antes[antes.Fecha >= INICIO_42H], por=('Canal',))['WAPE_%'],
    '42 h depurado':      resumen(gb_final[gb_final.Fecha >= INICIO_42H], por=('Canal',))['WAPE_%'],
    'Sesgo depurado':     resumen(gb_final, por=('Canal',))['Sesgo_%']}).to_string())

# El GB depurado reemplaza al anterior en la tabla de resultados
resultados = pd.concat([resultados[resultados.modelo != 'Gradient Boosting'], gb_final], ignore_index=True)

Picking Packing  · es_quincena             : aporte máximo +0.62 puntos → se queda
Picking Packing  · dias_operativos_semana  : aporte máximo -0.17 puntos → se quita
TAT              · es_quincena             : aporte máximo +0.00 puntos → se quita
TAT              · dias_operativos_semana  : aporte máximo +0.07 puntos → se quita
TAT              · dia_mes                 : aporte máximo +2.37 puntos → se queda
{'Picking Packing': 19, 'TAT': 21} variables finales

Gradient Boosting: con todas las candidatas frente a las variables depuradas
                 Ajuste (depurado)  May-nov antes  May-nov depurado  42 h antes  42 h depurado  Sesgo depurado
Canal                                                                                                         
Picking Packing               23.4           23.6              23.3        21.7           21.4            -2.2
TAT                           19.1           26.2              26.1        18.5           18.4            -2.3


### Conclusiones: variables candidatas
| Variable | PP | TAT | Por qué |
|---|---|---|---|
| `es_quincena` | ✅ **Se queda** (hasta +0.6 puntos) | ❌ Se quita (0.0) | En PP aporta en las tres ventanas. En TAT, su información ya está dentro de `dia_mes` (los días 1, 2, 15, 16, 30 y 31) |
| `dias_operativos_semana` | ❌ Se quita (quitarla **mejora**) | ❌ Se quita (+0.07, ruido) | No aporta en ningún canal. Coincide con el ARIMA (p = 0.14 en PP y 0.86 en TAT) y con la importancia por permutación (≈0) |
| `dia_mes` | — | ✅ **Se queda** (+2.4 puntos en el régimen actual) | Es la candidata que más aporta |

- **`dia_mes` en TAT confirma el hallazgo del notebook 02:** en el periodo de ajuste (2025 a abril de 2026) no aporta nada, pero en el régimen actual quitarla sube el error **+2.4 puntos**. Es el **patrón nuevo de 2026**: TAT sube a lo largo del mes desde mayo. El ARIMA no lo detectó (p = 0.70) porque lo trata como una línea recta; el árbol sí lo capta, porque el efecto **no es lineal**. Sigue siendo una pregunta para la operación: ¿hay un empuje comercial de cierre de mes en TAT?
- **`es_quincena` en PP:** pequeña pero constante (hasta +0.6 puntos). Coincide con Prophet y el ARIMA, que le asignan −61 a −72 líneas por día.
- **Modelo final más simple, sin perder precisión:** **19 variables en PP** (de 20) y **21 en TAT** (de 23). El WAPE queda igual o un poco mejor:
  - **PP:** 23.3% en may-nov y **21.4%** en el régimen actual.
  - **TAT:** 26.1% en may-nov y **18.4%** en el régimen actual.
- **Sesgo:** −2.2% en PP y −2.3% en TAT. Subestima levemente; se tiene en cuenta en la elección (sección 10).
- **Alcance:** la depuración se aplica al **Gradient Boosting**. Prophet y el ARIMA se dejan con todas sus variables: sus coeficientes muestran que `dias_operativos_semana` y `dia_mes` tienen un efecto casi nulo en ellos, así que quitarlas no cambiaría sus resultados de forma relevante.

## 10. Comparación final y elección del modelo

Se comparan los **seis modelos** con los mismos 29 cortes y los mismos días:
- Las dos líneas base (naive y promedio de 4 semanas).
- **Ridge (lineal):** la versión honesta del Experimento 0, que se incluye como el modelo lineal clásico del curso (pipeline con OneHotEncoder y StandardScaler).
- ARIMA (regresión), Prophet y Gradient Boosting (versión depurada de la sección 9).

**Qué se mira:**
1. **WAPE en dos ventanas:** may-nov 2026 y **jornada de 42 h** (régimen actual, el más parecido a como se usará el modelo).
2. **Sesgo:** subestimar deja la operación corta de personal.
3. **Error del volumen semanal:** la planeación de personal se hace por semana, así que importa cuánto se equivoca el **total de la semana**, no solo cada día.
4. **Prueba bootstrap por semanas** :* con 29 semanas, una diferencia de 1-2 puntos puede ser suerte. El bootstrap simula 4,000 "historias alternativas" re-muestreando semanas completas y calcula un **intervalo del 90%** para la diferencia frente a la línea base. Si el intervalo incluye el 0, la diferencia **no se distingue del ruido**.

**Regla de elección (fijada al inicio del proyecto):** se elige el modelo con menor WAPE en el régimen actual **solo si le gana a la línea base con evidencia** (bootstrap) y **no subestima más de 3%**. Si ningún modelo lo logra, se recomienda el promedio de 4 semanas.

In [31]:
# ============================================
# CELDA 10.1: Tabla final de todos los modelos
# ============================================
# --- El Ridge del Experimento 0 (versión honesta) entra como sexto modelo ---
ridge = pd.concat([backtest('Ridge (lineal)', hacer_ridge(con_trampa=False), c) for c in COL], ignore_index=True)
resultados = pd.concat([resultados[resultados.modelo != 'Ridge (lineal)'], ridge], ignore_index=True)

ORDEN_MODELOS = ['Naive (lag_7)', 'Promedio 4 semanas', 'Ridge (lineal)', 'ARIMA (regresión)',
                 'Prophet', 'Gradient Boosting']
actual = resultados[resultados.Fecha >= INICIO_42H]

def wape_semanal(df):
    """Error del VOLUMEN de la semana (suma lun-vie), que es lo que usa la planeación de personal."""
    s = df.groupby(['modelo', 'Canal', 'corte'])[['real', 'pred']].sum().reset_index()
    return s.groupby(['Canal', 'modelo']).apply(lambda g: metricas(g.real, g.pred)['WAPE_%']).round(1)

final = pd.concat({
    'WAPE may-nov':   resumen(resultados, por=('Canal', 'modelo'))['WAPE_%'],
    'WAPE 42 h':      resumen(actual, por=('Canal', 'modelo'))['WAPE_%'],
    'MAE 42 h':       resumen(actual, por=('Canal', 'modelo'))['MAE'],
    'Sesgo may-nov':  resumen(resultados, por=('Canal', 'modelo'))['Sesgo_%'],
    'Sesgo 42 h':     resumen(actual, por=('Canal', 'modelo'))['Sesgo_%'],
    'WAPE semanal 42 h': wape_semanal(actual),
}, axis=1)
final = final.reindex([(c, m) for c in COL for m in ORDEN_MODELOS])
print('Comparación final · backtesting de 29 semanas por canal\n')
print(final.to_string())

# --- Gráfica: WAPE de cada modelo en las dos ventanas ---
g = final[['WAPE may-nov', 'WAPE 42 h']].stack().rename('WAPE_%').reset_index()
g.columns = ['Canal', 'Modelo', 'Ventana', 'WAPE_%']
fig = px.bar(g, x='WAPE_%', y='Modelo', color='Ventana', facet_col='Canal', barmode='group', orientation='h',
             text='WAPE_%', color_discrete_sequence=['#b0b0b0', '#2a78d6'],
             category_orders={'Modelo': ORDEN_MODELOS[::-1]},
             title='WAPE de cada modelo: may-nov 2026 y régimen actual (jornada de 42 h)')
fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.update_layout(height=480, width=1100, legend=dict(orientation='h', y=1.12), xaxis_title='WAPE %',
                  xaxis2_title='WAPE %', yaxis_title='')
fig.for_each_annotation(lambda a: a.update(text=a.text.replace('Canal=', '')))
fig.show()

Comparación final · backtesting de 29 semanas por canal

                                    WAPE may-nov  WAPE 42 h  MAE 42 h  Sesgo may-nov  Sesgo 42 h  WAPE semanal 42 h
Canal           modelo                                                                                             
Picking Packing Naive (lag_7)               31.7       26.7     206.0           -1.5        -0.6               16.6
                Promedio 4 semanas          24.3       21.3     164.4           -1.4        -0.2               14.6
                Ridge (lineal)              23.7       22.4     172.5           -0.2         2.6               13.9
                ARIMA (regresión)           24.1       23.3     179.9            2.1         4.4               14.5
                Prophet                     24.1       23.4     180.7            1.5         4.4               13.9
                Gradient Boosting           23.3       21.4     165.5           -2.2        -1.4               13.2
TAT            

In [32]:
# ============================================
# CELDA 10.2: ¿Las diferencias son reales o ruido? Bootstrap por semanas
# ============================================
def bootstrap_vs_base(df, modelo, canal, base='Promedio 4 semanas', B=4000, semilla=42):
    """Diferencia de WAPE (modelo − línea base) y su intervalo del 90 % re-muestreando SEMANAS completas."""
    a = df[(df.modelo == modelo) & (df.Canal == canal)].set_index(['corte', 'Fecha'])
    b = df[(df.modelo == base) & (df.Canal == canal)].set_index(['corte', 'Fecha']).reindex(a.index)
    sem = pd.DataFrame({'ea': (a.real - a.pred).abs(), 'eb': (b.real - b.pred).abs(), 'y': a.real}).groupby('corte').sum()
    ea, eb, y = sem.ea.to_numpy(), sem.eb.to_numpy(), sem.y.to_numpy()
    idx = np.random.default_rng(semilla).integers(0, len(sem), size=(B, len(sem)))   # B "historias" alternativas
    dif = 100 * (ea[idx].sum(1) - eb[idx].sum(1)) / y[idx].sum(1)
    lo, hi = np.percentile(dif, [5, 95])
    return 100 * (ea.sum() - eb.sum()) / y.sum(), lo, hi, len(sem)

filas = []
for ventana, df in [('May-nov', resultados), ('Jornada 42 h', actual)]:
    for canal in COL:
        for modelo in ORDEN_MODELOS[2:]:
            d, lo, hi, n = bootstrap_vs_base(df, modelo, canal)
            veredicto = ('MEJOR que la base' if hi < 0 else 'PEOR que la base' if lo > 0 else 'no distinguible')
            filas.append({'Ventana': ventana, 'Canal': canal, 'Modelo': modelo, 'Diferencia': round(d, 1),
                          'IC 90% desde': round(lo, 1), 'IC 90% hasta': round(hi, 1), 'Semanas': n,
                          'Veredicto': veredicto})
boot = pd.DataFrame(filas).set_index(['Ventana', 'Canal', 'Modelo'])
print('Diferencia de WAPE frente al Promedio 4 semanas (puntos; negativo = el modelo es mejor)\n')
print(boot.to_string())

Diferencia de WAPE frente al Promedio 4 semanas (puntos; negativo = el modelo es mejor)

                                                Diferencia  IC 90% desde  IC 90% hasta  Semanas        Veredicto
Ventana      Canal           Modelo                                                                             
May-nov      Picking Packing Ridge (lineal)           -0.6          -2.6           1.3       29  no distinguible
                             ARIMA (regresión)        -0.3          -3.0           2.3       29  no distinguible
                             Prophet                  -0.3          -3.0           2.4       29  no distinguible
                             Gradient Boosting        -1.0          -3.0           1.0       29  no distinguible
             TAT             Ridge (lineal)           -3.1          -6.6           0.4       29  no distinguible
                             ARIMA (regresión)        -0.4          -3.6           2.7       29  no distinguible
       

In [33]:
# ============================================
# CELDA 10.3: Elección del modelo por canal
# ============================================
SESGO_MAX = -3.0      # % : subestimar más que esto deja la operación corta de personal

ELEGIDO = {}
for canal in COL:
    b = boot.xs(('Jornada 42 h', canal))
    f = final.xs(canal)
    # Regla: el modelo debe ser MEJOR que la base con evidencia (bootstrap) y no subestimar más de 3 %
    ganadores = [m for m in b.index if b.loc[m, 'Veredicto'] == 'MEJOR que la base'
                 and f.loc[m, 'Sesgo 42 h'] >= SESGO_MAX]
    ELEGIDO[canal] = min(ganadores, key=lambda m: f.loc[m, 'WAPE 42 h']) if ganadores else 'Promedio 4 semanas'
    retador = f.loc[ORDEN_MODELOS[2:], 'WAPE 42 h'].idxmin()          # el mejor modelo con variables
    print(f'{canal}')
    print(f'  Modelo elegido : {ELEGIDO[canal]}  (WAPE 42 h {f.loc[ELEGIDO[canal], "WAPE 42 h"]}%, '
          f'sesgo {f.loc[ELEGIDO[canal], "Sesgo 42 h"]:+.1f}%)')
    print(f'  Mejor retador  : {retador}  (WAPE 42 h {f.loc[retador, "WAPE 42 h"]}%, sesgo {f.loc[retador, "Sesgo 42 h"]:+.1f}%, '
          f'diferencia {b.loc[retador, "Diferencia"]:+.1f} puntos, IC 90% [{b.loc[retador, "IC 90% desde"]}, '
          f'{b.loc[retador, "IC 90% hasta"]}] → {b.loc[retador, "Veredicto"]})')
    mae = f.loc[ELEGIDO[canal], 'MAE 42 h']
    print(f'  Error típico del elegido: {mae:.0f} líneas por día ≈ {mae / LINEAS_POR_OPERARIO[canal]:.1f} operarios\n')

Picking Packing
  Modelo elegido : Promedio 4 semanas  (WAPE 42 h 21.3%, sesgo -0.2%)
  Mejor retador  : Gradient Boosting  (WAPE 42 h 21.4%, sesgo -1.4%, diferencia +0.1 puntos, IC 90% [-1.8, 2.1] → no distinguible)
  Error típico del elegido: 164 líneas por día ≈ 1.6 operarios

TAT
  Modelo elegido : Promedio 4 semanas  (WAPE 42 h 20.5%, sesgo +0.1%)
  Mejor retador  : Gradient Boosting  (WAPE 42 h 18.4%, sesgo -5.6%, diferencia -2.1 puntos, IC 90% [-4.8, 0.8] → no distinguible)
  Error típico del elegido: 317 líneas por día ≈ 1.3 operarios



### 10.4 ¿Y en un periodo con eventos?
La ventana de evaluación (may-nov 2026) **no tiene ningún evento grande**: ni cierre de año ni Semana Santa. Es el terreno más favorable para el promedio de 4 semanas, que no sabe que viene un evento.

Para ver qué pasa cuando sí hay eventos, se comparan los modelos en el **periodo de ajuste** (ene-2025 a abr-2026), que incluye el cierre de año 2025, la Semana Santa de 2025 y 2026, y los festivos de todo un año. No hay que correr nada nuevo: se reutilizan los pronósticos que ya se calcularon en las celdas 7.1 y 8.1 para elegir el ARIMA y el GB.

**Precaución:** este periodo se usó para elegir la configuración de cada modelo (no para compararlos con la base), así que su ventaja puede estar un poco inflada. Por eso solo se considera una diferencia relevante si el bootstrap la separa claramente del cero.

In [36]:
# ============================================
# CELDA 10.4: ¿Y en un periodo CON eventos? Comparación en los cortes de ajuste (ene-2025 a abr-2026)
# ============================================
# Se reutilizan los pronósticos que ya se calcularon para elegir el ARIMA (7.1) y el GB (8.1)
partes = []
for canal in COL:
    arima_elegido = next(n for n, o in ORDENES.items() if o == ORDEN_ELEGIDO[canal])
    partes += [ajuste[(ajuste.Canal == canal) & (ajuste.modelo == 'Promedio 4 semanas')],
               ajuste[(ajuste.Canal == canal) & (ajuste.modelo == arima_elegido)].assign(modelo='ARIMA (regresión)'),
               ajuste_gb[(ajuste_gb.Canal == canal) & (ajuste_gb.modelo == CONFIG_GB[canal])]
                   .assign(modelo='Gradient Boosting')]
aj = pd.concat(partes, ignore_index=True)
aj = aj.merge(datos[['Fecha', 'Canal'] + EVENTOS], on=['Fecha', 'Canal'])
aj['Tipo'] = np.where(aj[EVENTOS].sum(axis=1) > 0, 'Día con evento', 'Día normal')

# --- 1. ¿Le ganan a la base con evidencia? (bootstrap por semanas) ---
filas = []
for canal in COL:
    for modelo in ['ARIMA (regresión)', 'Gradient Boosting']:
        d, lo, hi, n = bootstrap_vs_base(aj, modelo, canal)
        veredicto = ('MEJOR que la base' if hi < 0 else 'PEOR que la base' if lo > 0 else 'no distinguible')
        filas.append({'Canal': canal, 'Modelo': modelo, 'Diferencia': round(d, 1), 'IC 90% desde': round(lo, 1),
                      'IC 90% hasta': round(hi, 1), 'Semanas': n, 'Veredicto': veredicto})
print(f'Periodo de ajuste ({aj.Fecha.min().date()} a {aj.Fecha.max().date()}): diferencia de WAPE frente al '
      'Promedio 4 semanas\n')
print(pd.DataFrame(filas).set_index(['Canal', 'Modelo']).to_string())

# --- 2. ¿De dónde sale la diferencia? Días con evento frente a días normales ---
print('\nWAPE % por tipo de día (cierre de año, pre-cierre, reactivación, Semana Santa y semana siguiente)\n')
tipo = resumen(aj, por=('Canal', 'modelo', 'Tipo'))[['WAPE_%', 'n']].unstack('Tipo')
print(tipo.to_string())

Periodo de ajuste (2025-01-07 a 2026-05-02): diferencia de WAPE frente al Promedio 4 semanas

                                   Diferencia  IC 90% desde  IC 90% hasta  Semanas          Veredicto
Canal           Modelo                                                                               
Picking Packing ARIMA (regresión)        -5.8         -10.0          -2.5       35  MEJOR que la base
                Gradient Boosting        -5.9          -9.1          -3.4       35  MEJOR que la base
TAT             ARIMA (regresión)         0.1          -1.9           2.0       34    no distinguible
                Gradient Boosting        -2.9          -4.5          -1.3       34  MEJOR que la base

WAPE % por tipo de día (cierre de año, pre-cierre, reactivación, Semana Santa y semana siguiente)

                                           WAPE_%                         n           
Tipo                               Día con evento Día normal Día con evento Día normal
Canal           mode

### Conclusiones: comparación final y elección
**Decisión: un esquema mixto según el tipo de semana.**
| Tipo de semana | Modelo recomendado | Por qué |
|---|---|---|
| **Semanas normales** (la mayor parte del año) | **Promedio de 4 semanas** | Ningún modelo le gana con evidencia en may-nov 2026; es simple, transparente y sin sesgo |
| **Semanas con eventos** (15-dic a 20-ene; Semana Santa y la semana siguiente) | **Gradient Boosting** (en PP, también sirve el ARIMA) | En el periodo con eventos le gana a la base con evidencia estadística |

**En may-nov 2026 (sin eventos):**
- **Todos los modelos con variables quedan a ±3 puntos de la línea base**, y el bootstrap dice que **ninguna diferencia se distingue del azar** con 29 semanas (19 en el régimen actual).
- **En el régimen actual, el promedio de 4 semanas es el mejor de PP** (21.3%, sesgo −0.2%). El Gradient Boosting empata (21.4%).
- **En TAT, el Gradient Boosting es el retador más fuerte:** 18.4% contra 20.5% (−2.1 puntos) y el **mejor error semanal (12.0% contra 14.4%)**. No cumple la regla porque su ventaja no es concluyente (intervalo −4.8 a +0.8) y **subestima −5.6%**, más allá del límite de −3%.
- **Prophet y el ARIMA no superan al Ridge**, el modelo lineal simple: la complejidad adicional no se tradujo en precisión.

**En el periodo con eventos (ene-2025 a abr-2026), la historia cambia (sección 10.4):**
- **PP:** el ARIMA y el GB le ganan a la base por **5.8 puntos**, con evidencia (intervalos lejos del cero).
- **TAT:** el GB le gana por **2.9 puntos**, con evidencia. El ARIMA empata.
- **La ventaja viene de los días con evento:** en PP, el promedio de 4 semanas falla **88%** en cierre de año y Semana Santa, contra ≈50% de los modelos. El promedio no sabe que viene un evento; los modelos sí, por las variables del notebook 02.
- **También ganan en días normales** de ese periodo, aunque por menos (PP: 21.5-21.8% contra 25.2%).

**La lectura de conjunto:** el resultado de may-nov no significa que los modelos no sirvan, sino que **esa ventana no tenía eventos**, que es donde los modelos aportan. La jornada de 42 h no cambió esto: lo que hizo fue dejar poca historia del régimen actual (19 semanas), lo que hace menos concluyente cualquier comparación reciente.

**Error típico del promedio de 4 semanas en el régimen actual:** **164 líneas por día en PP (≈1.6 operarios)** y **317 en TAT (≈1.3 operarios)**.

⚠️ **Urgente:** el próximo cierre de año empieza en ≈3 semanas. Es el periodo en que el promedio de 4 semanas falla más y donde el GB debería usarse.

## 11. Pronóstico de la semana del 23 al 27 de noviembre de 2026 y operarios necesarios
*Conversión a operarios y alerta del martes basadas en el notebook de modelos base del equipo.*

Se simula el uso real del modelo: con los datos hasta el **viernes 20 de noviembre**, se pronostican los 5 días operativos de la semana siguiente usando `features_proxima_semana.csv`, que no tiene `Lineas` porque esos días todavía no han ocurrido.

**Qué se entrega por día y canal:**
| Columna | Qué es |
|---|---|
| **Pronóstico** | Promedio de 4 semanas, el modelo elegido en la sección 10 |
| **Rango probable (80%)** | Sale de los errores reales del backtesting en el régimen actual: en 8 de cada 10 días, el volumen real cayó dentro de ese rango |
| **GB retador** | Pronóstico del Gradient Boosting, para comparar en paralelo (no se usa para programar) |
| **Operarios** | Pronóstico ÷ capacidad por operario (106 líneas/día en PP, 249 en TAT), **redondeado hacia arriba** |
| **Operarios con el máximo del rango** | Cuántos harían falta si el día sale en el extremo alto |
| **Semanas post-festivo** | Cuántas de las 4 semanas que se promedian fueron un día después de festivo (alerta de pronóstico "contaminado") |

In [34]:
# ============================================
# CELDA 11: Pronóstico de la semana del 23 al 27 de noviembre de 2026 y operarios
# ============================================
DIAS = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves', 4: 'Viernes'}
prox = proxima.copy()

# --- 1. Modelo elegido: promedio de 4 semanas (mismo respaldo del backtesting) ---
prox['Pronostico'] = respaldo(prox, ['prom_4_semanas', 'nivel_reciente'])

# --- 2. Rango probable (80 %) a partir de los errores reales del backtesting en el régimen actual ---
base_42h = resultados[(resultados.modelo == 'Promedio 4 semanas') & (resultados.Fecha >= INICIO_42H)]
rel = (base_42h.real / base_42h.pred).groupby(base_42h.Canal).quantile([0.10, 0.90]).unstack()
prox['Minimo_80'] = prox.Pronostico * prox.Canal.map(rel[0.10])
prox['Maximo_80'] = prox.Pronostico * prox.Canal.map(rel[0.90])

# --- 3. Retador: Gradient Boosting depurado, entrenado con TODA la historia ---
for canal in COL:
    d = datos[datos.Canal == canal].dropna(subset=['nivel_reciente'])
    cols = FEATURES_GB[canal]
    params, _ = CANDIDATAS_GB[CONFIG_GB[canal]]
    m = HistGradientBoostingRegressor(categorical_features=[cols.index(c) for c in CATEGORICAS_GB],
                                      random_state=42, **params).fit(d[cols], d[OBJETIVO])
    fila = prox.Canal == canal
    prox.loc[fila, 'GB_retador'] = np.clip(m.predict(prox.loc[fila, cols]), 0, None)

# --- 4. Alerta: ¿alguna de las 4 semanas que se promedian fue un día después de festivo? ---
post = datos.set_index(['Canal', 'Fecha'])['dia_despues_festivo']
for k in (7, 14, 21, 28):
    prox[f'post_{k}'] = [post.get((c, f - pd.Timedelta(days=k)), 0) for c, f in zip(prox.Canal, prox.Fecha)]
prox['Semanas_post_festivo'] = prox[[f'post_{k}' for k in (7, 14, 21, 28)]].sum(axis=1)

# --- 5. Operarios: líneas ÷ capacidad, redondeado hacia ARRIBA (no se programa media persona) ---
prox['Capacidad'] = prox.Canal.map(LINEAS_POR_OPERARIO)
prox['Operarios'] = np.ceil(prox.Pronostico / prox.Capacidad).astype(int)
prox['Operarios_max_80'] = np.ceil(prox.Maximo_80 / prox.Capacidad).astype(int)
prox['Día'] = prox.Fecha.dt.dayofweek.map(DIAS)

tabla = prox[['Canal', 'Fecha', 'Día', 'Pronostico', 'Minimo_80', 'Maximo_80', 'GB_retador',
              'Operarios', 'Operarios_max_80', 'Semanas_post_festivo']]
print('Pronóstico de la semana del 23 al 27 de noviembre de 2026 (corte: viernes 20 de noviembre)\n')
print(tabla.round(0).to_string(index=False))

print('\nResumen de la semana por canal')
print(prox.groupby('Canal').agg(Lineas_semana=('Pronostico', 'sum'), GB_semana=('GB_retador', 'sum'),
                                Operarios_dia_prom=('Operarios', 'mean'), Operarios_max=('Operarios', 'max'))
          .round(1).to_string())

# --- 6. Gráfica: últimas 8 semanas reales y el pronóstico con su rango ---
for canal in COL:
    h = datos[(datos.Canal == canal) & (datos.Fecha > datos.Fecha.max() - pd.Timedelta(days=56))]
    p = prox[prox.Canal == canal]
    fig = go.Figure()
    fig.add_scatter(x=h.Fecha, y=h[OBJETIVO], name='Real (últimas 8 semanas)', mode='lines+markers',
                    line=dict(color='#9a9a9a', width=1.5), marker=dict(size=5))
    fig.add_scatter(x=list(p.Fecha) + list(p.Fecha[::-1]), y=list(p.Maximo_80) + list(p.Minimo_80[::-1]),
                    fill='toself', fillcolor=COL[canal], opacity=0.15, line=dict(width=0), name='Rango probable (80 %)')
    fig.add_scatter(x=p.Fecha, y=p.Pronostico, name='Promedio 4 semanas (elegido)', mode='lines+markers',
                    line=dict(color=COL[canal], width=3))
    fig.add_scatter(x=p.Fecha, y=p.GB_retador, name='Gradient Boosting (retador)', mode='lines+markers',
                    line=dict(color='#333333', width=1.5, dash='dot'))
    fig.update_layout(title=f'{canal}: pronóstico de la semana del 23 al 27 de noviembre', height=420, width=1100,
                      yaxis_title='Líneas por día', xaxis_title='', legend=dict(orientation='h', y=1.15))
    fig.show()

Pronóstico de la semana del 23 al 27 de noviembre de 2026 (corte: viernes 20 de noviembre)

          Canal      Fecha       Día  Pronostico  Minimo_80  Maximo_80  GB_retador  Operarios  Operarios_max_80  Semanas_post_festivo
Picking Packing 2026-11-23     Lunes       898.0      587.0     1199.0       671.0          9                12                     0
Picking Packing 2026-11-24    Martes       545.0      356.0      727.0       705.0          6                 7                     2
Picking Packing 2026-11-25 Miércoles       969.0      633.0     1293.0       933.0         10                13                     0
Picking Packing 2026-11-26    Jueves       837.0      546.0     1116.0       934.0          8                11                     0
Picking Packing 2026-11-27   Viernes       745.0      487.0      994.0       840.0          8                10                     0
            TAT 2026-11-23     Lunes      1444.0      983.0     1895.0      1479.0          6           

### Conclusiones: pronóstico de la semana del 23 al 27 de noviembre
| Canal | Líneas en la semana | Operarios por día (promedio) | Máximo |
|---|---|---|---|
| Picking Packing | ≈3,995 | 8.2 | 10 (miércoles) |
| TAT | ≈7,219 | 6.2 | 7 (jueves y viernes) |

**⚠️ Alerta del martes 24 (los dos canales): el pronóstico está "contaminado".** Dos de los cuatro martes que se promedian fueron **días después de un festivo** (3 y 17 de noviembre, después de los lunes festivos del 2 y el 16). El martes 24 **no** es día después de festivo, así que se espera un martes normal:
- **TAT:** el promedio da **1,078 líneas (5 operarios)**, pero un martes normal del régimen actual tiene ≈**1,510-1,535 líneas** (mediana y promedio de 13 martes). En TAT, los martes después de festivo salen muy bajos (mediana 769). **Recomendación: programar 7 operarios** (1,535 ÷ 249 = 6.2 → 7). El GB retador, que sí conoce `dia_despues_festivo`, pronostica 2,140: va en la misma dirección, aunque más alto.
- **PP:** el promedio da 545 líneas (6 operarios), por debajo de un martes normal (≈650-690). En PP, los martes después de festivo no salen más bajos, así que la contaminación pesa menos. Aun así, conviene programar **7 operarios**, con el máximo del rango (727) como referencia.

**Rango probable:** en el régimen actual, el volumen real cae entre ≈65% y ≈134% del pronóstico en 8 de cada 10 días. Si la operación quiere cubrir el extremo alto, necesita entre 1 y 3 operarios más por día (columna `Operarios_max_80`). Es la decisión de cuánto "colchón" tener, y le corresponde a la operación.

**El GB retador y el promedio coinciden en PP** (3,995 contra 4,154 líneas en la semana, +4%), **pero no en TAT** (7,219 contra 8,750, +21%), sobre todo por el martes. Seguir comparándolos semana a semana es justamente la prueba en paralelo recomendada en la sección 10.

**Lección para la operación:** el efecto de un festivo **se contagia a los pronósticos de las 4 semanas siguientes** a través del promedio. Una mejora simple y de bajo costo es **excluir del promedio los días después de festivo** (o ajustarlos manualmente), como se hizo aquí con el martes 24.

## 12. Conclusiones generales, limitaciones y próximos pasos

### Pregunta del proyecto
¿Qué modelo pronostica mejor las **líneas de picking por día y canal** de la semana siguiente, con la información disponible el viernes en la noche?

### Resultado
**Se recomienda un esquema mixto:**
- **Semanas normales:** promedio de las últimas 4 semanas (mismo día de la semana). En may-nov 2026, una ventana sin eventos, ningún modelo le gana con una diferencia que se distinga del ruido.
- **Semanas con eventos** (15-dic a 20-ene; Semana Santa y la semana siguiente): **Gradient Boosting**. En un periodo con eventos (ene-2025 a abr-2026), le gana a la base con evidencia estadística: −5.8 puntos en PP y −2.9 en TAT.

| Modelo | PP may-nov | PP jornada 42 h | TAT may-nov | TAT jornada 42 h |
|---|---|---|---|---|
| Naive (lag_7) | 31.7% | 26.7% | 32.5% | 25.8% |
| **Promedio 4 semanas (elegido)** | 24.3% | **21.3%** | 26.5% | 20.5% |
| Ridge (lineal) | 23.7% | 22.4% | **23.4%** | 20.7% |
| ARIMA (regresión) | 24.1% | 23.3% | 26.1% | 23.1% |
| Prophet | 24.1% | 23.4% | 24.2% | 22.0% |
| Gradient Boosting | **23.3%** | 21.4% | 26.1% | **18.4%** |

*WAPE: porcentaje del volumen pronosticado mal. Menor es mejor.*

### Hallazgos principales
1. **En semanas normales, una regla simple bien construida es difícil de superar.** El promedio de 4 semanas se equivoca en ≈**164 líneas por día en PP (≈1.6 operarios)** y ≈**317 en TAT (≈1.3 operarios)**, sin sesgo. Con 19 semanas del régimen actual, los modelos con variables no logran una ventaja demostrable en esas semanas.
2. **En semanas con eventos, los modelos sí ganan, y con claridad.** En cierre de año y Semana Santa, el promedio de 4 semanas se equivoca en **88% en PP**, contra ≈50% del ARIMA y el GB. El valor de los modelos está en **anticipar lo que la historia reciente no muestra**.
3. **Cada modelo gana en un momento distinto:**
   - Los modelos con variables de régimen (Ridge, Prophet) ganan **cuando hay cambios** (may-jul TAT, recorte de rutas).
   - El promedio de 4 semanas gana **en semanas estables**.
   - El Gradient Boosting gana **cuando el régimen ya está asentado** (TAT desde agosto).
4. **El Gradient Boosting es el retador más fuerte en TAT** (18.4% contra 20.5% en el régimen actual; 12.0% contra 14.4% en el volumen semanal), pero su ventaja no es concluyente y **subestima −5.6%**, más allá del límite fijado.
5. **Dónde falla la regla simple** (y cómo corregirlo):
   - **Cambios de régimen:** tarda ≈4 semanas en reaccionar (mayo TAT: sobreestimó +31%).
   - **Festivos:** el día después de festivo en TAT falla 62% y "contamina" el promedio de las 4 semanas siguientes (martes 24-nov).
   - **Lunes de PP:** se subestiman −7% desde la jornada de 42 h.
6. **Los efectos de calendario son coherentes entre modelos:** Prophet y el ARIMA llegan a los mismos efectos por caminos distintos. Por ejemplo, el día después de festivo en TAT resta ≈390 líneas, el cierre de año ≈−790 a −925 y el recorte de rutas ≈−550 a −575.
7. **El data leakage habría engañado al proyecto:** con las variables del mismo día, el error parecía bajar de ≈24% a ≈7% (Experimento 0). Ese modelo no se puede usar en la operación.
8. **Variables depuradas:** `dias_operativos_semana` no aporta en ningún canal; `es_quincena` aporta en PP; `dia_mes` aporta en TAT (patrón nuevo de 2026, no lineal).

### Pronóstico de la semana del 23 al 27 de noviembre
| Canal | Líneas en la semana | Operarios por día (promedio) | Máximo |
|---|---|---|---|
| Picking Packing | ≈3,995 | 8.2 | 10 |
| TAT | ≈7,219 | 6.2 | 7 |

⚠️ **Martes 24:** ajustar a un martes normal (≈7 operarios en TAT), porque el promedio está contaminado por dos martes después de festivo.

### Supuestos y limitaciones
- **Poca historia del régimen actual:** solo **19 semanas** de la jornada de 42 h (87 días por canal). Por eso las diferencias entre modelos no son concluyentes y el ranking podría cambiar con más datos. Tampoco incluye todavía un **diciembre** bajo este régimen.
- **Regímenes conocidos desde el inicio:** en el backtesting se supone que la operación sabe de una decisión de negocio (recorte de rutas, catálogo) desde el día en que entra en vigencia. Si se espera 15 días a que los datos la reflejen, el error de TAT en mayo sube de 39% a 58% (sección 7.4).
- **Se pronostican días operativos normales:** los sábados bajo planeación, domingos y festivos operados quedan fuera; si se programan, se estiman aparte.
- **Se modelan líneas procesadas, no demanda total:** si hubo pedidos que no se alcanzaron a alistar, no están en los datos.
- **Capacidad por operario fija** (106 líneas/día en PP, 249 en TAT): es el promedio del régimen actual (EDA). Variaciones de productividad cambian los operarios necesarios.
- **Reproducibilidad:** Prophet, ARIMA y Gradient Boosting pueden variar unas décimas con otras versiones de las librerías; las conclusiones no cambian.
- **Pocos eventos:** Semana Santa, cierre de año y días después de festivo ocurren pocas veces en 4 años (11 a 60 días). Sus efectos son coherentes entre modelos, pero tienen incertidumbre.
- **La ventana de evaluación principal (may-nov 2026) no tiene eventos grandes.** Favorece a la regla simple. La ventaja de los modelos en semanas con eventos se midió en el periodo de ajuste, que también se usó para elegir la configuración de cada modelo, así que puede estar un poco inflada. Aun así, las diferencias (−5.8 y −2.9 puntos) están claramente separadas del cero.

### Próximos pasos sugeridos
1. **Implementar el esquema mixto ya, empezando por el cierre de año 2026** (desde el 15 de diciembre): usar el Gradient Boosting en esas semanas y comparar su pronóstico con el real día a día.
2. **En semanas normales, usar el promedio de 4 semanas con tres ajustes de bajo costo:** excluir o corregir los días después de festivo, ajustar manualmente desde el primer día de un cambio de régimen y agregar un margen a los lunes de PP.
3. **Correr el Gradient Boosting en paralelo en TAT** también en semanas normales, durante 8-12 semanas. Si mantiene su ventaja (≈2 puntos) y se corrige su sesgo, podría usarse todo el año.
4. **Reevaluar en marzo de 2027,** con el primer cierre de año bajo la jornada de 42 h y más semanas del régimen actual.
5. **Conseguir la información que más mejoraría el pronóstico:** qué **pedidos y rutas están programados** para la semana siguiente antes del corte del viernes.
6. **Validar con el área comercial** el patrón de TAT a lo largo del mes (`dia_mes`): ¿hay un empuje de cierre de mes?